<a href="https://colab.research.google.com/github/KatreenGhobrial/RepoCloudComputing/blob/main/HW/HW3/HW3_GIRAFFE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""

######
1. **Setup & Configuration**
2. **Data Layer**
3. **Backend / Business Logic**
4. **UI Components**
5. **Frontend Application**
6. **QA Tests**
7. **Launch**

##1. Setup & Configuration
"""

# Install all runtime dependencies used by the notebook.
!pip -q install pandas numpy scikit-learn "gradio==5.49.1" nltk markdown google-generativeai pillow matplotlib requests beautifulsoup4
!pip -q install transformers safetensors gdown

from pathlib import Path
import zipfile
import shutil
import gdown
import re

# Paste your public Google Drive link here
DRIVE_URL = "https://drive.google.com/file/d/1LUOB1Imt4wppi38QQh5Rm5JH6d8PeO1W/view?usp=sharing"

ZIP_PATH = Path("/content/hf_basil_health_classifier_new.zip")
HF_MODEL_DIR = Path("/content/hf_basil_health_classifier_new")
EXTRACT_DIR = Path("/content/hf_model_extracted")


def extract_drive_file_id(url):
    patterns = [
        r"/file/d/([^/]+)",
        r"id=([^&]+)"
    ]

    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return match.group(1)

    raise ValueError("Could not extract Google Drive file ID from the link.")


file_id = extract_drive_file_id(DRIVE_URL)
download_url = f"https://drive.google.com/uc?id={file_id}"

if ZIP_PATH.exists():
    print("Model zip already downloaded, skipping re-download:", ZIP_PATH)
else:
    print("Downloading model zip from Google Drive...")
    try:
        gdown.download(download_url, str(ZIP_PATH), quiet=False)
    except Exception as exc:
        raise RuntimeError(
            "Failed to download the model zip from Google Drive. This is usually "
            "either a lost connection or Google Drive throttling the file after too "
            "many downloads today. Wait a few minutes and re-run this cell, or "
            "re-upload the zip to Drive and update DRIVE_URL."
        ) from exc

if not ZIP_PATH.exists():
    raise FileNotFoundError("Model zip was not downloaded.")

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

# Find the real HuggingFace model folder automatically
config_files = list(EXTRACT_DIR.rglob("config.json"))

if not config_files:
    raise FileNotFoundError(
        "Could not find config.json inside the zip. "
        "Make sure the zip contains a HuggingFace model folder."
    )

real_model_dir = config_files[0].parent

if HF_MODEL_DIR.exists():
    shutil.rmtree(HF_MODEL_DIR)

shutil.copytree(real_model_dir, HF_MODEL_DIR)

print("HuggingFace model is ready:", HF_MODEL_DIR)

import os
import re
import json
import time
import html as html_lib
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import gradio as gr
from bs4 import BeautifulSoup
try:
    import markdown as markdown_lib
except ImportError:
    markdown_lib = None
import PIL.Image

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.stem import PorterStemmer

SentenceTransformer = None
TRANSFORMERS_AVAILABLE = False

try:
    import google.generativeai as genai
    GEMINI_AVAILABLE = True
except ImportError:
    genai = None
    GEMINI_AVAILABLE = False


def render_markdown(text: str) -> str:
    if markdown_lib is not None:
        return markdown_lib.markdown(text)
    return html_lib.escape(str(text)).replace("\n", "<br>")


def load_gemini_api_key() -> Optional[str]:
    """Read the API key from Colab Secrets first, then from an environment variable."""
    key = os.getenv("GEMINI_API_KEY", "").strip()

    try:
        from google.colab import userdata
        colab_key = userdata.get("GEMINI_API_KEY")
        if colab_key:
            key = str(colab_key).strip()
    except Exception:
        pass

    return key or None


GEMINI_API_KEY = load_gemini_api_key()

if GEMINI_AVAILABLE and GEMINI_API_KEY:
    genai.configure(api_key=GEMINI_API_KEY)
    vision_model = genai.GenerativeModel("gemini-2.5-flash")
else:
    vision_model = None

print("Gemini:", "configured" if vision_model else "not configured — local fallback remains available")
print("Embeddings: TF-IDF only (Hugging Face ignored)")

"""##2. Data Layer"""

DEFAULT_BASIL_PAPERS = [
       {
        "title": "Chemical Composition, Antioxidant and Antimicrobial Activities of Basil (Ocimum basilicum) Essential Oils Depends on Seasonal Variations",
        "authors": (
            "Abdullah Ijaz Hussain, Farooq Anwar, "
            "Syed Tufail Hussain Sherazi, Roman Przybylski"
        ),
        "journal": "Food Chemistry",
        "year": 2008,
        "doi": "10.1016/j.foodchem.2007.12.010",
        "url": "https://www.sciencedirect.com/science/article/abs/pii/S0308814607012666",
        "abstract": (
            "The study examined basil essential oils collected during summer, "
            "autumn, winter, and spring. Essential-oil yield, chemical "
            "composition, antioxidant activity, and antimicrobial activity "
            "changed significantly between seasons. Linalool was the dominant "
            "compound, while winter samples contained more oxygenated "
            "monoterpenes. The oils inhibited all tested bacterial and fungal "
            "microorganisms."
        )
    },

    {
        "title": "Basil Downy Mildew (Peronospora belbahrii): Discoveries and Challenges Relative to Its Control",
        "authors": (
            "Christian A. Wyenandt, James E. Simon, Robert M. Pyne, "
            "Kathryn Homa, Margaret T. McGrath, Shouan Zhang, "
            "Richard N. Raid, Li-Jun Ma, Robert Wick, Li Guo, "
            "Angela Madeiras"
        ),
        "journal": "Phytopathology",
        "year": 2015,
        "doi": "10.1094/PHYTO-02-15-0032-FI",
        "url": "https://doi.org/10.1094/PHYTO-02-15-0032-FI",
        "abstract": (
            "This review examines basil downy mildew caused by "
            "Peronospora belbahrii, a major threat to sweet basil production. "
            "The disease can spread through contaminated seed and is difficult "
            "to control because many commercial sweet basil varieties lack "
            "genetic resistance. The paper reviews breeding programs, pathogen "
            "biology, cultural management, and conventional and organic "
            "fungicide treatments."
        )
    },

    {
        "title": "Effects of Green Synthesized Zinc and Copper Nano-Fertilizers on the Morphological and Biochemical Attributes of Basil Plant",
        "authors": (
            "Ahmadreza Abbasifar, Fatemeh Shahrabadi, "
            "Babak ValizadehKaji"
        ),
        "journal": "Journal of Plant Nutrition",
        "year": 2020,
        "doi": "10.1080/01904167.2020.1724305",
        "url": "https://www.tandfonline.com/doi/full/10.1080/01904167.2020.1724305",
        "abstract": (
            "Zinc and copper nanoparticles were produced through green "
            "synthesis using basil extract and were applied to basil plants "
            "at different concentrations. Selected combinations improved "
            "plant growth, chlorophyll and carotenoid concentrations, phenolic "
            "and flavonoid contents, and antioxidant activity. The results "
            "show that the biological effects depend strongly on the "
            "concentrations of both nanoparticles."
        )
    },

    {
        "title": "Alteration in Light Spectra Causes Opposite Responses in Volatile Phenylpropanoids and Terpenoids Compared with Phenolic Acids in Sweet Basil (Ocimum basilicum) Leaves",
        "authors": (
            "Minna Kivimäenpää, Adedayo Mofikoya, "
            "Ahmed M. Abd El-Raheem, Johanna Riikonen, "
            "Riitta Julkunen-Tiitto, Jarmo K. Holopainen"
        ),
        "journal": "Journal of Agricultural and Food Chemistry",
        "year": 2022,
        "doi": "10.1021/acs.jafc.2c03309",
        "url": "https://pmc.ncbi.nlm.nih.gov/articles/PMC9545148/",
        "abstract": (
            "Sweet basil plants were grown under three different LED light "
            "spectra to examine changes in growth, leaf anatomy, photosynthesis, "
            "and secondary metabolites. Light spectra that increased volatile "
            "terpenoids and phenylpropanoids tended to reduce phenolic acids "
            "and plant growth, while spectra that promoted growth and phenolic "
            "acids reduced several volatile compounds. Glandular trichome "
            "density was associated with terpenoid and eugenol production."
        )
    },
   {
    "title": "First Report of Bacterial Leaf Spot Caused by Pseudomonas cichorii on Thai Basil in Hawaii",
    "authors": (
        "Blaine C. Luiz, Wade P. Heller, Eva Brill, "
        "Brian C. Bushe, Lisa M. Keith"
    ),
    "journal": "Plant Disease",
    "year": 2018,
    "doi": "10.1094/PDIS-06-18-0995-PDN",
    "url": "https://apsjournals.apsnet.org/doi/10.1094/PDIS-06-18-0995-PDN",
    "abstract": (
        "Water-soaked, irregular, gray-to-black leaf spots were observed "
        "on Thai basil plants. The lesions ranged from small spots to larger "
        "necrotic areas near the leaf margins. Laboratory isolation and "
        "pathogenicity testing identified Pseudomonas cichorii as the cause "
        "of the bacterial leaf spot disease."
    )
}
]




# --------------------------------------------------
# Dynamic URL-based article loading for the RAG workflow
# --------------------------------------------------
# The list above is only a source list. At notebook startup we fetch the
# article text from each URL once, save it locally, and then build the
# inverted index and RAG from the fetched text.
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
PAPERS_FILE_PATH = DATA_DIR / "basil_papers_from_urls.json"


def fetch_article_text(url: str, max_chars: int = 12000) -> str:
    """Read article text from a URL using simple web scraping."""
    if not url:
        return ""

    try:
        response = requests.get(
            url,
            timeout=12,
            headers={"User-Agent": "Mozilla/5.0"}
        )
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        for tag in soup(["script", "style", "nav", "footer", "header"]):
            tag.decompose()

        chunks = []
        for tag in soup.find_all(["h1", "h2", "h3", "p"]):
            text = tag.get_text(" ", strip=True)
            if len(text) >= 40:
                chunks.append(text)

        return " ".join(chunks)[:max_chars]

    except Exception as exc:
        print(f"Could not read URL: {url}")
        print(f"Reason: {exc}")
        return ""


def load_papers_from_urls(
    sources: List[Dict[str, Any]] = DEFAULT_BASIL_PAPERS
) -> List[Dict[str, Any]]:
    """Create paper objects from URL content once at startup."""
    papers = []

    for source in sources:
        url = source.get("url", "")
        web_text = fetch_article_text(url)

        # URL content is the main source. The old abstract is only a safety
        # fallback when an academic site blocks scraping.
        content = web_text or source.get("abstract", "")

        if not content.strip():
            continue

        papers.append({
            "title": source.get("title", "Unknown"),
            "authors": source.get("authors", "Unknown"),
            "journal": source.get("journal", "Unknown"),
            "year": source.get("year", ""),
            "doi": source.get("doi", ""),
            "url": url,
            "abstract": content,
            "source_type": "url" if web_text else "fallback_abstract"
        })

    return papers


def save_papers_to_file(
    papers: List[Dict[str, Any]],
    file_path: Path = PAPERS_FILE_PATH
) -> str:
    file_path.parent.mkdir(parents=True, exist_ok=True)

    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(list(papers or []), f, ensure_ascii=False, indent=2)

    return str(file_path)


def load_papers_from_file(
    file_path: Path = PAPERS_FILE_PATH
) -> List[Dict[str, Any]]:
    with open(file_path, "r", encoding="utf-8") as f:
        papers = json.load(f)

    return [
        paper for paper in papers
        if isinstance(paper, dict) and str(paper.get("abstract", "")).strip()
    ]


def add_paper_url_to_file(
    title: str,
    url: str,
    authors: str = "Unknown"
) -> int:
    papers = load_papers_from_file()
    text = fetch_article_text(url)

    if not text:
        raise ValueError("Could not read text from this URL.")

    papers.append({
        "title": title.strip() or "Unknown",
        "authors": authors.strip() or "Unknown",
        "url": url.strip(),
        "abstract": text,
        "source_type": "url"
    })

    save_papers_to_file(papers)
    return len(papers)


# Run once at notebook startup:
# 1. Read all article URLs.
# 2. Save the fetched text to JSON.
# 3. The following cells build the inverted index and RAG from this loaded data.
basil_papers = load_papers_from_urls()
save_papers_to_file(basil_papers)

print(f"Loaded {len(basil_papers)} papers from article URLs.")
print(f"Saved fetched article text to {PAPERS_FILE_PATH}")

for paper in basil_papers:
    print("-", paper["title"], "| source:", paper.get("source_type", "unknown"))

STOP_WORDS = {
    'a', 'about', 'above', 'after', 'again', 'against', 'all', 'am',
    'an', 'and', 'any', 'are', 'aren', 'as', 'at', 'be', 'because',
    'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by',
    'can', 'could', 'couldn', 'did', 'didn', 'do', 'does', 'doesn',
    'doing', 'don', 'down', 'during', 'each', 'few', 'for', 'from',
    'further', 'had', 'hadn', 'has', 'hasn', 'have', 'haven', 'having',
    'he', 'her', 'here', 'hers', 'herself', 'him', 'himself', 'his',
    'how', 'i', 'if', 'in', 'into', 'is', 'isn', 'it', 'its', 'itself',
    'just', 'll', 'm', 'ma', 'me', 'might', 'more', 'most', 'mustn',
    'my', 'myself', 'needn', 'no', 'nor', 'now', 'of', 'off',
    'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves',
    'out', 'over', 'own', 're', 's', 'same', 'shan', 'she', 'should',
    'shouldn', 'so', 'some', 'such', 't', 'than', 'that', 'the',
    'their', 'theirs', 'them', 'themselves', 'then', 'there', 'these',
    'they', 'this', 'those', 'through', 'to', 'too', 'under', 'until',
    'up', 've', 'very', 'was', 'wasn', 'we', 'were', 'weren', 'what',
    'when', 'where', 'which', 'while', 'who', 'whom', 'why', 'will',
    'with', 'won', 'would', 'wouldn', 'y', 'you', 'your', 'yours',
    'yourself', 'yourselves'
}

TARGET_TERMS = {
    'terpenoids', 'phenolic', 'spectra', 'trichome', 'nanoparticles',
    'chlorophyll', 'flavonoid', 'antioxidant', 'peronospora', 'mildew',
    'seed', 'fungicide', 'essential', 'antimicrobial', 'linalool',
    'composition', 'pseudomonas', 'spot', 'lesion', 'necrotic'
}

stemmer = PorterStemmer()


def build_target_inverted_index(
    papers: List[Dict[str, Any]],
    target_terms: set[str] = TARGET_TERMS
) -> Dict[str, Dict[str, Any]]:
    """Build the tutorial-style term index: total count and source URLs."""
    index: Dict[str, Dict[str, Any]] = {}
    target_stems = {stemmer.stem(term): term for term in target_terms}

    for paper in papers:
        text = f"{paper.get('title', '')} {paper.get('abstract', '')}".lower()
        paper_url = paper.get("url", "")

        for word in re.findall(r"\w+", text):
            if word in STOP_WORDS:
                continue

            stemmed_word = stemmer.stem(word)
            if word in target_terms or stemmed_word in target_stems:
                saved_term = word if word in target_terms else target_stems[stemmed_word]
                item = index.setdefault(saved_term, {"term": saved_term,"DocIDs": [],"count": 0})
                item["count"] += 1
                if paper_url and paper_url not in item["DocIDs"]:
                    item["DocIDs"].append(paper.get("url", ""))

    return index


inverted_index = build_target_inverted_index(basil_papers)


print("=== TARGET TERM INDEX ===")
for term in sorted(inverted_index):
    item = inverted_index[term]
    print(f"\nterm: {item['term']}")
    print(f"count: {item['count']}")
    print("DocIDs:")

    for doc_id in item["DocIDs"]:
        print(f"  - {doc_id}")


# --------------------------------------------------
# Firebase DB export layer
# --------------------------------------------------
FIREBASE_DB_URL = "https://cloudroot-9da6c-default-rtdb.europe-west1.firebasedatabase.app/"
FIREBASE_INDEX_PATH = "basil_target_index"
LOCAL_FIREBASE_EXPORT_PATH = DATA_DIR / "firebase_index_export.json"


def build_firebase_index_payload(index: Dict[str, Dict[str, Any]]) -> Dict[str, Any]:
    payload = {}
    for term, item in sorted(index.items()):
        safe_key = re.sub(r"[^A-Za-z0-9_-]", "_", term)
        payload[safe_key] = {
            "term": item["term"],
            "count": int(item["count"]),
            "DocIDs": list(item["DocIDs"]),
        }
    return payload


def upload_index_to_firebase(index: Dict[str, Dict[str, Any]]) -> Dict[str, Any]:
    payload = build_firebase_index_payload(index)

    if FIREBASE_DB_URL:
        url = f"{FIREBASE_DB_URL}/{FIREBASE_INDEX_PATH}.json"
        response = requests.put(url, json=payload, timeout=20)
        response.raise_for_status()
        return {
            "status": "uploaded_to_firebase",
            "url": url,
            "terms": len(payload),
        }

    with open(LOCAL_FIREBASE_EXPORT_PATH, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

    return {
        "status": "saved_local_firebase_export",
        "file": str(LOCAL_FIREBASE_EXPORT_PATH),
        "terms": len(payload),
    }


def load_index_from_firebase_or_local() -> Dict[str, Any]:
    if FIREBASE_DB_URL:
        url = f"{FIREBASE_DB_URL}/{FIREBASE_INDEX_PATH}.json"
        response = requests.get(url, timeout=20)
        response.raise_for_status()
        return response.json() or {}

    if LOCAL_FIREBASE_EXPORT_PATH.exists():
        with open(LOCAL_FIREBASE_EXPORT_PATH, "r", encoding="utf-8") as f:
            return json.load(f)

    return build_firebase_index_payload(inverted_index)


firebase_sync_result = upload_index_to_firebase(inverted_index)
print("Firebase index sync:", firebase_sync_result)

"""##3. Backend / Business Logic"""

class SimpleVectorStore:

    def __init__(self):
        self.documents: List[str] = []
        self.embeddings: List[Any] = []
        self.metadatas: List[Dict[str, Any]] = []
        self.ids: List[str] = []

    def clear(self) -> None:
        self.documents.clear()
        self.embeddings.clear()
        self.metadatas.clear()
        self.ids.clear()

    def add(self, embeddings, documents, metadatas, ids) -> None:
        self.embeddings.extend(np.asarray(embeddings))
        self.documents.extend(documents)
        self.metadatas.extend(metadatas)
        self.ids.extend(ids)

    def query(self, query_embeddings, n_results: int = 3) -> Dict[str, List[List[Any]]]:
        if not self.embeddings:
            return {
                "ids": [[]],
                "documents": [[]],
                "metadatas": [[]],
                "distances": [[]],
            }

        similarities = cosine_similarity(
            np.asarray(query_embeddings),
            np.asarray(self.embeddings)
        )[0]

        result_count = min(max(int(n_results), 1), len(self.documents))
        top_indices = np.argsort(similarities)[::-1][:result_count]

        return {
            "ids": [[self.ids[i] for i in top_indices]],
            "documents": [[self.documents[i] for i in top_indices]],
            "metadatas": [[self.metadatas[i] for i in top_indices]],
            "distances": [[float(1 - similarities[i]) for i in top_indices]],
        }


class EcologicalRAG:

    def __init__(
        self,
        gemini_api_key: Optional[str] = None,
        use_transformers: bool = True
    ):
        self.collection = SimpleVectorStore()
        self.papers: List[Dict[str, Any]] = []
        self.fitted = False
        self.use_transformers = False
        self.embedding_model = None
        self.tfidf = TfidfVectorizer(max_features=1000, stop_words="english")

        if use_transformers and TRANSFORMERS_AVAILABLE:
            try:
                self.embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
                self.use_transformers = True
            except Exception as exc:
                print(f"SentenceTransformer fallback to TF-IDF: {exc}")

        self.use_gemini = bool(gemini_api_key and GEMINI_AVAILABLE)
        self.text_model = None

        if self.use_gemini:
            genai.configure(api_key=gemini_api_key)
            self.text_model = genai.GenerativeModel("gemini-2.5-flash")

    @staticmethod
    def preprocess_text(text: str) -> str:
        if not text:
            return ""
        text = re.sub(r"\s+", " ", text)
        text = re.sub(r"[^\w\s\-\.]", " ", text)
        return text.strip()

    @staticmethod
    def _split_sentences(text: str) -> List[str]:
        """Split an abstract into clean sentences without using a language model."""
        clean_text = re.sub(r"\s+", " ", str(text or "")).strip()
        if not clean_text:
            return []

        sentences = re.split(r"(?<=[.!?])\s+", clean_text)
        return [
            sentence.strip()
            for sentence in sentences
            if len(re.findall(r"\w+", sentence)) >= 4
        ]

    def generate_embeddings(self, texts: List[str]):
        if self.use_transformers:
            return self.embedding_model.encode(texts, show_progress_bar=False)

        if not self.fitted:
            self.tfidf.fit(texts)
            self.fitted = True

        return self.tfidf.transform(texts).toarray()
    #load papers
    def load_papers(self, papers_data: List[Dict[str, Any]]) -> None:
        valid_papers = [p for p in papers_data if p.get("abstract", "").strip()]
        if not valid_papers:
            raise ValueError("No papers with abstracts were supplied.")

        documents = [
            self.preprocess_text(f"{paper.get('title', '')}. {paper.get('abstract', '')}")
            for paper in valid_papers
        ]
        metadatas = [
            {
                "title": paper.get("title", "Unknown"),
                "authors": paper.get("authors", "Unknown"),
                "url": paper.get("url", ""),
            }
            for paper in valid_papers
        ]
        ids = [f"paper_{i}" for i in range(len(valid_papers))]

        self.collection.clear()
        self.papers = list(valid_papers)

        # TF-IDF must be refitted whenever the document collection changes.
        self.fitted = False
        self.tfidf = TfidfVectorizer(max_features=1000, stop_words="english")
        embeddings = self.generate_embeddings(documents)

        self.collection.add(
            embeddings=embeddings,
            documents=documents,
            metadatas=metadatas,
            ids=ids,
        )

    def search(self, query: str, n_results: int = 3):
        query_processed = self.preprocess_text(query)
        if not query_processed:
            return {
                "ids": [[]],
                "documents": [[]],
                "metadatas": [[]],
                "distances": [[]],
            }

        query_embedding = self.generate_embeddings([query_processed])
        return self.collection.query(query_embedding, n_results=n_results)

    def _rank_relevant_sentences(

          self,
          query_text: str,
          relevant_papers: List[Dict[str, Any]],
          max_sentences: int = 4
          ) -> List[Dict[str, Any]]:
          """
          Select sentences according to the number of query words
          that appear in each sentence.
          """

          # Prepare the query words using stop words and stemming
          query_words = {
              stemmer.stem(word)
              for word in re.findall(r"\w+", query_text.lower())
              if word not in STOP_WORDS
          }

          ranked_sentences = []

          # Go over the retrieved papers
          for paper_number, paper in enumerate(relevant_papers, start=1):

              sentences = self._split_sentences(
                  paper.get("abstract", "")
              )

              for sentence in sentences:

                  sentence_words = {
                      stemmer.stem(word)
                      for word in re.findall(r"\w+", sentence.lower())
                      if word not in STOP_WORDS
                  }

                  # Count common words between the query and the sentence
                  score = len(query_words.intersection(sentence_words))

                  if score > 0:
                      ranked_sentences.append({
                          "sentence": sentence,
                          "paper_number": paper_number,
                          "score": score
                      })

          # Sort from the highest score to the lowest
          ranked_sentences.sort(
              key=lambda item: item["score"],
              reverse=True
          )

          # If no matching sentence was found, use the first sentence
          # from the highest ranked paper
          if not ranked_sentences and relevant_papers:
              first_sentences = self._split_sentences(
                  relevant_papers[0].get("abstract", "")
              )

              if first_sentences:
                  ranked_sentences.append({
                      "sentence": first_sentences[0],
                      "paper_number": 1,
                      "score": 0
                  })

          return ranked_sentences[:max_sentences]


    def _local_grounded_response(
                self,
                query_text: str,
                relevant_papers: List[Dict[str, Any]]
            ) -> str:
            """
            Build a simple answer from sentences found in the retrieved papers.
            """

            if not relevant_papers:
                return (
                    "**Research-based answer**\n\n"
                    "No relevant paper was found."
                )

            selected_sentences = self._rank_relevant_sentences(
                query_text,
                relevant_papers
            )

            lines = [
                "**Research-based answer**",
                "",
                f"**Question:** {query_text}",
                "",
            ]

            used_sources = []

            for item in selected_sentences:
                source_number = item["paper_number"]

                lines.append(
                    f"- {item['sentence']} "
                    f"**[Source {source_number}]**"
                )

                if source_number not in used_sources:
                    used_sources.append(source_number)

            lines.extend(["", "**Sources used:**"])

            for source_number in used_sources:
                paper = relevant_papers[source_number - 1]

                source_line = (
                    f"{source_number}. "
                    f"**{paper.get('title', 'Unknown')}** — "
                    f"{paper.get('authors', 'Unknown')}"
                )

                if paper.get("url"):
                    source_line += (
                        f" ([Open article]({paper['url']}))"
                    )

                lines.append(source_line)

            return "\n".join(lines)

    def query(
                self,
                query_text: str,
                n_results: int = 3
            ) -> Dict[str, Any]:
                """Retrieve relevant papers and create an answer."""

                # Find the most relevant papers
                search_results = self.search(
                    query_text,
                    n_results=n_results
                )
                distances = search_results.get("distances", [[]])[0]

                similarities = [1 - distance for distance in distances]

                MIN_SIMILARITY = 0.20

                # relevant_indices = [
                #     i for i, similarity in enumerate(similarities)
                #     if similarity >= MIN_SIMILARITY
                # ]

                relevant_indices = []
                for i, similarity in enumerate(similarities):
                    if similarity >= MIN_SIMILARITY:
                        relevant_indices.append(i)

                if not relevant_indices:
                    return {
                        "response": (
                            "I could not find relevant information in the basil research papers "
                            "for this question."
                        ),
                        "papers_found": 0,
                        "papers": [],
                        "search_results": search_results,
                        "mode": "NO_RELEVANT_RESULTS",
                    }
                relevant_papers = []

                for doc_id in search_results["ids"][0]:
                    paper_index = int(doc_id.split("_")[1])
                    relevant_papers.append(
                        self.papers[paper_index]
                    )

                # The default answer is the local summary
                response_mode = "extractive"
                fallback_reason = None

                # Use Gemini only if it is configured
                if self.use_gemini and self.text_model is not None:
                    context_parts = []

                    for source_number, paper in enumerate(
                        relevant_papers,
                        start=1
                    ):
                        document = search_results["documents"][0][
                            source_number - 1
                        ]

                        context_parts.append(
                            f"[Source {source_number}]\n"
                            f"Title: {paper.get('title', 'Unknown')}\n"
                            f"Authors: {paper.get('authors', 'Unknown')}\n"
                            f"URL: {paper.get('url', '')}\n"
                            f"Research content: {document}"
                        )

                    context_text = "\n\n".join(context_parts)

                    prompt = f"""
                          You are an agriculture and basil-plant research assistant.

                          Answer the question using only the supplied research sources.

                          Rules:
                          - Do not invent information.
                          - Cite claims using [Source 1], [Source 2], and so on.
                          - If the sources do not contain enough information, say so.
                          - Give a clear and concise answer.
                          - Answer in the same language as the user's question.

                          Question:
                          {query_text}

                          Research sources:
                          {context_text}
                          """

                    try:
                        response = self.text_model.generate_content(
                            prompt,
                            generation_config=genai.types.GenerationConfig(
                                temperature=0.3,
                                max_output_tokens=800,
                            ),request_options={"timeout": 15},
                        )

                        response_text = str(response.text or "" ).strip()

                        if not response_text:raise RuntimeError("Gemini returned an empty response.")

                        response_mode = "gemini"

                    except Exception as exc:
                        fallback_reason = str(exc)
                        response_text = self._local_grounded_response(query_text, relevant_papers)

                else:
                    fallback_reason = "Gemini is not configured."

                    response_text = self._local_grounded_response(query_text,relevant_papers)

                return {
                    "response": response_text,
                    "response_mode": response_mode,
                    "fallback_reason": fallback_reason,
                    "papers_found": len(relevant_papers),
                    "papers": relevant_papers,
                    "search_results": search_results,
                }

USE_GENERATIVE_AI_FOR_RAG = True

rag_system = EcologicalRAG(
    gemini_api_key=(GEMINI_API_KEY if USE_GENERATIVE_AI_FOR_RAG else None),
    use_transformers=False,
)


def reload_research_sources() -> int:
    """Manual refresh: reload papers from JSON and rebuild the Boolean index and RAG once."""
    global basil_papers, inverted_index
    basil_papers = load_papers_from_file()
    inverted_index = build_target_inverted_index(basil_papers)
    rag_system.load_papers(basil_papers)
    return len(basil_papers)


papers_count = reload_research_sources()
print(f"RAG service ready. Index was built once from {papers_count} papers in {PAPERS_FILE_PATH}.")
print(
    "Article summary mode:",
    "Gemini" if rag_system.use_gemini else "TF-IDF extractive template (no generative AI)",
)

def build_paper_index(paper: Dict[str, Any]) -> Dict[str, int]:
    index: Dict[str, int] = {}
    text = f"{paper.get('title', '')} {paper.get('abstract', '')}".lower()

    for word in re.findall(r"\w+", text):
        if word in STOP_WORDS:
            continue
        stemmed_word = stemmer.stem(word)
        index[stemmed_word] = index.get(stemmed_word, 0) + 1

    return index


def prepare_query_words(query: str) -> List[str]:
    return [
        stemmer.stem(word)
        for word in re.findall(r"\w+", query.lower())
        if word not in STOP_WORDS and word not in {"and", "or"}
    ]


def boolean_search(query: str, mode: str = "AUTO", limit: int = 3) -> List[Dict[str, Any]]:
    query = query.strip()
    if not query:
        return []

    normalized_mode = mode.upper()
    if normalized_mode == "AUTO":
        lowered = f" {query.lower()} "
        normalized_mode = "AND" if " and " in lowered else "OR"

    clean_query = re.sub(r"\b(?:AND|OR)\b", " ", query, flags=re.IGNORECASE)
    query_words = prepare_query_words(clean_query)
    if not query_words:
        return []

    matched_papers = []

    for paper in basil_papers:
        paper_index = build_paper_index(paper)
        found_words = {
            word: paper_index[word]
            for word in query_words
            if word in paper_index
        }

        is_match = (
            len(found_words) == len(set(query_words))
            if normalized_mode == "AND"
            else bool(found_words)
        )

        if is_match:
            result = dict(paper)
            result["found_words"] = found_words
            result["score"] = sum(found_words.values())
            result["mode"] = normalized_mode
            matched_papers.append(result)

    matched_papers.sort(key=lambda item: item["score"], reverse=True)
    return matched_papers[:limit]

# def boolean_search(query: str, mode: str = "AUTO", limit: int = 3) -> List[Dict[str, Any]]:
#     query = query.strip()
#     if not query:
#         return []

#     normalized_mode = mode.upper()
#     if normalized_mode == "AUTO":
#         lowered = f" {query.lower()} "
#         normalized_mode = "AND" if " and " in lowered else "OR"

#     clean_query = re.sub(r"\b(?:AND|OR)\b", " ", query, flags=re.IGNORECASE)
#     query_words = prepare_query_words(clean_query)
#     if not query_words:
#         return []

#     matched_papers = []

#     for paper in basil_papers:
#         paper_url = paper.get("url", "")
#         found_words = {}

#         for word in query_words:
#             matched_key = None
#             if word in inverted_index:
#                 matched_key = word
#             else:
#                 for key in inverted_index:
#                     if stemmer.stem(key) == word:
#                         matched_key = key
#                         break

#             if matched_key:
#                 if paper_url in inverted_index[matched_key]["DocIDs"]:
#                     found_words[matched_key] = 1

#         is_match = (
#             len(found_words) == len(set(query_words))
#             if normalized_mode == "AND"
#             else bool(found_words)
#         )

#         if is_match:
#             result = dict(paper)
#             result["found_words"] = found_words
#             result["score"] = len(found_words)
#             result["mode"] = normalized_mode
#             matched_papers.append(result)

#     matched_papers.sort(key=lambda item: item["score"], reverse=True)
#     return matched_papers[:limit]


def advanced_html_search(query: str, mode: str = "AUTO"):
    if not query or not query.strip():
        yield "<div style='color:#ef4444; padding:20px;'>Please enter a search term.</div>"
        return

    yield """
    <div style="text-align:center; padding:40px;">
        <div style="font-size:40px; margin-bottom:10px;">🔍</div>
        <h3 style="color:#0645ad;">Scanning Knowledge Base...</h3>
    </div>
    """

    matched_papers = boolean_search(query, mode)


    if not matched_papers:
        papers_html = f"""
        <div style="text-align:center; padding:30px; color:var(--muted);">
            <h3>No exact Boolean Search matches for “{html_lib.escape(query)}”.</h3>
        </div>
        """
    else:
        papers_html = (
            f"<div style='color:var(--muted); margin-bottom:20px;'>"
            f"Found {len(matched_papers)} Boolean result(s).</div>"
        )

        for paper in matched_papers:
            found_words_text = ", ".join(
                f"{html_lib.escape(word)}: {count}"
                for word, count in paper["found_words"].items()
            )
            papers_html += f"""
            <div class="research-paper-card" style="background:#ffffff; border:1px solid var(--border);
                        border-radius:12px; padding:20px; margin-bottom:16px;">
                <div style="font-size:18px; font-weight:700; color:#087a24;">
                    {html_lib.escape(paper.get("title", "Unknown"))}
                </div>
                <div style="font-size:13px; color:var(--muted); margin:8px 0 12px;">
                    {html_lib.escape(paper.get("authors", "Unknown"))}
                </div>
                <div style="font-size:14px; color:#111827; line-height:1.6;">
                    {html_lib.escape(paper.get("abstract", ""))}
                </div>
                <div style="margin-top:14px; color:#92400e; font-size:12px;">
                    Score: {paper["score"]} · Found: {found_words_text}
                </div>
                <div style="margin-top:12px;">
                    <a href="{html_lib.escape(paper.get('url', '#'))}" target="_blank"
                       style="color:#0645ad;">Open source ↗</a>
                </div>
            </div>
            """

    yield """
    <div style="text-align:center; padding:30px; color:#fbbf24;">
        <div style="font-size:32px;">📚 ⏳</div>
        <h3>Building a research-based summary...</h3>
    </div>
    """ + papers_html

   #!!!!!!!!!
    rag_result = rag_system.query(query, n_results=3)
    summary_html = render_markdown(rag_result["response"])

    is_ai_response = rag_result.get("response_mode") == "gemini"
    fallback_was_used = bool(rag_result.get("fallback_reason"))

    if is_ai_response:
        summary_title = "✨ AI-Enhanced Summary"
        summary_note = (
            "Generated with Gemini using the retrieved research papers."
        )

    elif fallback_was_used:
        summary_title = "📚 Research-Based Summary"
        summary_note = (
            "Gemini was unavailable, so the system automatically "
            "created a local summary from the retrieved papers."
        )

    else:
        summary_title = "📚 Research-Based Summary"
        summary_note = (
            "Built locally from the most relevant sentences "
            "in the retrieved papers — no generative AI."
        )

    final_html = f"""
    <div class="research-summary-card" style="background:#ffffff;
                border:1px solid var(--border); border-radius:16px;
                padding:24px; margin-bottom:24px; color:#111827;">
        <h3 style="color:#087a24; margin-top:0;">{summary_title}</h3>
        <div style="color:#374151; font-size:12px; margin-bottom:14px;">{summary_note}</div>
        <div class="research-summary-body" style="color:#111827; font-size:15px; line-height:1.7; -webkit-text-fill-color:#111827;">
            {summary_html}
        </div>
    </div>
    """

    yield final_html + papers_html


def search_and_wrap(query: str):
    for step in advanced_html_search(query):
        yield f"<div class='results-container'>{step}</div>"

def rag_question_answer(question: str, n_results: int = 3) -> str:
    if not question or not question.strip():
        return "<div class='page-card'>Please enter a question.</div>"

    result = rag_system.query(question, n_results=int(n_results))
    return f"""
    <div class="page-card">
        <div class="page-title">RAG Answer</div>
        <div class="page-subtitle">Mode: {html_lib.escape(str(result.get('response_mode', result.get('mode', 'unknown'))))}</div>
        <div style="color:var(--text); line-height:1.7; margin-top:14px;">
            {render_markdown(result.get('response', ''))}
        </div>
    </div>
    """

try:
    import torch
    from PIL import Image
    from transformers import AutoImageProcessor, AutoModelForImageClassification
    HF_CLASSIFIER_AVAILABLE = True
except Exception as hf_error:
    torch = None
    Image = None
    AutoImageProcessor = None
    AutoModelForImageClassification = None
    HF_CLASSIFIER_AVAILABLE = False
    print("HuggingFace classifier is not available:", hf_error)


LOCAL_CLASSIFIER_MODEL_PATH = str(HF_MODEL_DIR) if "HF_MODEL_DIR" in globals() else "/content/hf_basil_health_classifier_new"


class PlantImageClassifierService:
    """
    Local image-classification microservice.

    It loads the HuggingFace basil classifier once, then exposes
    a predict(image_path) method for the Plant Scanner UI.
    """

    def __init__(self, model_path: str = LOCAL_CLASSIFIER_MODEL_PATH):
        self.model_path = model_path
        self.processor = None
        self.model = None
        self.last_error = None

    def find_model_path(self):
        if os.path.exists(self.model_path):
            return self.model_path

        fallback_path = "/content/hf_basil_health_classifier_new"
        if os.path.exists(fallback_path):
            return fallback_path

        return None

    def load_model(self):
        if self.model is not None and self.processor is not None:
            return self.model

        if not HF_CLASSIFIER_AVAILABLE:
            raise RuntimeError("HuggingFace / Transformers is not available.")

        model_path = self.find_model_path()

        if model_path is None:
            raise FileNotFoundError(
                "Could not find the HuggingFace model folder. "
                "Run Cell 3 and upload the zip file first."
            )

        self.processor = AutoImageProcessor.from_pretrained(model_path)
        self.model = AutoModelForImageClassification.from_pretrained(model_path)
        self.model.eval()

        self.model_path = model_path
        self.last_error = None
        return self.model

    def is_healthy_label(self, label: str) -> bool:
        label = label.lower().replace("_", " ").replace("-", " ")

        unhealthy_words = [
            "not healthy",
            "nothealthy",
            "unhealthy",
            "bad",
            "disease",
            "infected",
            "sick"
        ]

        healthy_words = [
            "healthy",
            "good"
        ]

        if any(word in label for word in unhealthy_words):
            return False

        if any(word in label for word in healthy_words):
            return True

        # Default fallback if labels are unclear
        return False

    def predict(self, image_path: str) -> Dict[str, Any]:
        self.load_model()

        image = Image.open(image_path).convert("RGB")
        inputs = self.processor(images=image, return_tensors="pt")

        with torch.no_grad():
            outputs = self.model(**inputs)
            probs = torch.softmax(outputs.logits, dim=-1)[0]

        predicted_index = int(torch.argmax(probs).item())
        confidence = float(probs[predicted_index].item())

        id2label = self.model.config.id2label
        predicted_raw_label = str(id2label.get(predicted_index, predicted_index))

        is_healthy = self.is_healthy_label(predicted_raw_label)

        predicted_label = "Healthy" if is_healthy else "NotHealthy"
        status = "healthy" if is_healthy else "not_healthy"

        not_healthy_score = 0.0
        for i, prob in enumerate(probs):
            label = str(id2label.get(i, i))
            if not self.is_healthy_label(label):
                not_healthy_score += float(prob.item())

        return {
            "label": predicted_label,
            "raw_label": predicted_raw_label,
            "confidence": confidence,
            "raw_not_healthy_score": not_healthy_score,
            "status": status,
        }


plant_classifier_service = PlantImageClassifierService()

# ==========================================
# Plant Scanner Helper Functions
# ==========================================

import os
import json
import base64
import html as html_lib
import PIL.Image


try:
    plant_classifier_service.load_model()
    print("Local basil classifier loaded:", plant_classifier_service.model_path)
except Exception as exc:
    plant_classifier_service.last_error = str(exc)
    print("Local basil classifier is not loaded yet:", exc)


def get_uploaded_file_path(file_obj):
    """Extract file path from Gradio upload object."""
    if file_obj is None:
        return None

    if isinstance(file_obj, str):
        return file_obj

    if isinstance(file_obj, dict):
        for key in ("path", "name", "orig_name"):
            value = file_obj.get(key)
            if value:
                return value
        return None

    for attribute in ("name", "path"):
        value = getattr(file_obj, attribute, None)
        if value:
            return value

    return None


def normalize_uploaded_files(files):
    """Always return uploaded files as a list."""
    if files is None:
        return []

    if isinstance(files, (list, tuple)):
        return list(files)

    return [files]


def update_upload_preview(files):
    files = normalize_uploaded_files(files)

    image_paths = []
    for file_obj in files:
        path = get_uploaded_file_path(file_obj)
        if path and os.path.exists(path):
            image_paths.append(path)

    images_count = len(image_paths)

    status_text = "Uploaded Successfully" if images_count > 0 else "No Image Uploaded"
    status_color = "#30d158" if images_count > 0 else "#ffcc00"
    status_icon = "✅" if images_count > 0 else "⚠️"

    upload_status_html = f"""
    <div style="display:grid; grid-template-columns:repeat(3, 1fr);
                gap:18px; margin:20px 0 24px 0;">
        <div class="page-card">
            <div style="font-size:24px; margin-bottom:8px;">🖼️</div>
            <div style="color:#647067; font-size:12px; font-weight:800;">IMAGES UPLOADED</div>
            <div style="color:#1f2937; font-size:28px; font-weight:900; margin-top:6px;">{images_count}</div>
        </div>

        <div class="page-card">
            <div style="font-size:24px; margin-bottom:8px;">{status_icon}</div>
            <div style="color:#647067; font-size:12px; font-weight:800;">UPLOAD STATUS</div>
            <div style="color:{status_color}; font-size:20px; font-weight:900; margin-top:6px;">{status_text}</div>
        </div>

        <div class="page-card">
            <div style="font-size:24px; margin-bottom:8px;">🚀</div>
            <div style="color:#647067; font-size:12px; font-weight:800;">READY TO ANALYZE</div>
            <div style="color:#1f2937; font-size:28px; font-weight:900; margin-top:6px;">{images_count}</div>
        </div>
    </div>
    """

    imgs_html = ""

    for path in image_paths:
        try:
            with open(path, "rb") as f:
                b64 = base64.b64encode(f.read()).decode("utf-8")

            ext = os.path.splitext(path)[1].lower().replace(".", "")
            if ext == "jpg":
                ext = "jpeg"

            imgs_html += (
                f'<img src="data:image/{ext};base64,{b64}" '
                f'style="height:200px; object-fit:contain; '
                f'border-radius:8px; border:1px solid #d8e3d1; '
                f'background:#fff; margin:4px;" />'
            )

        except Exception:
            pass

    preview_html = f"""
    <div style="display:flex; flex-wrap:wrap; gap:8px; padding:12px;
                background:#ffffff; border:1px solid #d8e3d1;
                border-radius:12px; margin-top:8px;">
        {imgs_html}
    </div>
    """ if imgs_html else ""

    return upload_status_html, preview_html


def render_local_prediction_card(image_path, reason="Gemini fallback"):
    filename = html_lib.escape(os.path.basename(image_path))

    try:
        prediction = plant_classifier_service.predict(image_path)

        label = prediction.get("label", "Unknown")
        raw_label = prediction.get("raw_label", label)
        confidence = float(prediction.get("confidence", 0))
        raw_score = float(prediction.get("raw_not_healthy_score", 0))

        if label == "Healthy":
            status_label = "✅ Healthy"
            status_color = "#30d158"
            border_color = "rgba(48,209,88,.4)"
            diagnosis = "The local classifier predicts that the basil leaf is healthy."
            recommendations = [
                "Continue regular watering and sensor monitoring.",
                "Keep checking humidity and soil moisture from the dashboard.",
                "Upload another image if new spots or color changes appear.",
            ]
        else:
            status_label = "🟡 NotHealthy"
            status_color = "#ffcc00"
            border_color = "rgba(255,204,0,.4)"
            diagnosis = "The local classifier predicts that the basil leaf may be unhealthy."
            recommendations = [
                "Inspect the leaves closely and separate suspicious leaves if needed.",
                "Check humidity and soil moisture because high moisture can increase disease risk.",
                "Use Academic Search with terms such as mildew, spot, lesion, or peronospora.",
            ]

        recs_html = "".join(
            f"<li style='color:#111827;'>{html_lib.escape(item)}</li>"
            for item in recommendations
        )

        return f"""
        <div style="background:#ffffff; border:1px solid {border_color};
                    border-radius:14px; padding:18px; margin-top:14px;">
            <div style="display:flex; justify-content:space-between; flex-wrap:wrap; gap:10px;">
                <strong style="color:#111827;">{filename}</strong>
                <span style="color:{status_color}; font-weight:900;">{status_label}</span>
            </div>

            <p style="color:#111827; margin:10px 0 4px;">
                <strong style="color:#087a24;">Diagnosis:</strong> {diagnosis}
            </p>

            <p style="color:#555555; font-size:13px; margin:0 0 8px;">
                <strong style="color:#087a24;">Mode:</strong> Local HuggingFace classifier fallback<br>
                <strong style="color:#087a24;">Reason:</strong> {html_lib.escape(reason)}<br>
                <strong style="color:#087a24;">Raw model label:</strong> {html_lib.escape(str(raw_label))}<br>
                <strong style="color:#087a24;">Confidence:</strong> {confidence * 100:.2f}%<br>
                <strong style="color:#087a24;">Raw NotHealthy probability:</strong> {raw_score:.4f}
            </p>

            <div style="color:#111827;">
                <strong style="color:#087a24;">Recommendations:</strong>
                <ul style="margin-top:6px; padding-left:18px;">{recs_html}</ul>
            </div>
        </div>
        """

    except Exception as exc:
        return f"""
        <div style="background:#fff5f5; border:1px solid rgba(255,69,58,.4);
                    border-radius:14px; padding:18px; margin-top:14px;">
            <strong style="color:#ff453a;">⚠️ Local classifier failed for {filename}</strong>
            <p style="color:#555555;">{html_lib.escape(str(exc))}</p>
        </div>
        """


def run_local_classifier_analysis(files, reason="Gemini is not available"):
    files = normalize_uploaded_files(files)

    if not files:
        return "<div style='color:#ff453a; padding:10px;'>⚠️ Upload at least one image first.</div>"

    cards = []

    for file_obj in files:
        image_path = get_uploaded_file_path(file_obj)

        if image_path and os.path.exists(image_path):
            cards.append(render_local_prediction_card(image_path, reason=reason))

    if not cards:
        return "<div style='color:#ff453a; padding:10px;'>⚠️ Could not read the uploaded image files.</div>"

    return f"""
    <div class="page-card" style="margin-top:24px;">
        <h3 style="color:var(--green);">🌿 Local Basil Classifier Report</h3>
        <p style="color:var(--muted);">
            Gemini was not used for this result. The prediction comes from the trained HuggingFace basil model.
        </p>
        {''.join(cards)}
    </div>
    """


# def run_gemini_analysis(files):
#     files = normalize_uploaded_files(files)

#     if not files:
#         return "<div style='color:#ff453a; padding:10px;'>⚠️ Upload at least one image first.</div>"

#     current_vision_model = globals().get("vision_model", None)

#     if current_vision_model is None:
#         return run_local_classifier_analysis(
#             files,
#             reason="Gemini API is not configured"
#         )

#     reports = []

#     for index, file_obj in enumerate(files, start=1):
#         image_path = get_uploaded_file_path(file_obj)

#         if not image_path or not os.path.exists(image_path):
#             reports.append(
#                 f"<div style='color:#ff453a;'>⚠️ Image #{index}: could not read file path.</div>"
#             )
#             continue

#         try:
#             image = PIL.Image.open(image_path).convert("RGB")

#             prompt = (
#                 "You are an expert plant pathologist. Analyze this basil image. "
#                 "Return only valid JSON with keys: "
#                 "status (healthy, mild, or severe), "
#                 "diagnosis (short string), "
#                 "recommendations (list of exactly three actionable strings)."
#             )

#             response = current_vision_model.generate_content([prompt, image])

#             clean_text = (
#                 response.text
#                 .strip()
#                 .replace("```json", "")
#                 .replace("```", "")
#                 .strip()
#             )

#             result = json.loads(clean_text)

#             status = str(result.get("status", "unknown")).lower()
#             diagnosis = html_lib.escape(str(result.get("diagnosis", "No diagnosis.")))

#             recs = result.get("recommendations", [])
#             if not isinstance(recs, list):
#                 recs = [str(recs)]

#             status_map = {
#                 "healthy": ("✅ Healthy", "#30d158", "rgba(48,209,88,.3)"),
#                 "mild": ("🟡 Mild issue", "#ffcc00", "rgba(255,204,0,.3)"),
#                 "severe": ("🔴 Severe issue", "#ff453a", "rgba(255,69,58,.3)"),
#             }

#             label, color, border = status_map.get(
#                 status,
#                 ("⚪ Unknown", "#777777", "rgba(170,170,170,.3)")
#             )

#             recs_html = "".join(
#                 f"<li style='color:#111827;'>{html_lib.escape(str(item))}</li>"
#                 for item in recs[:3]
#             )

#             reports.append(f"""
#             <div style="background:#fff; border:1px solid {border}; border-radius:14px;
#                         padding:18px; margin-top:14px;">
#                 <div style="display:flex; justify-content:space-between; flex-wrap:wrap; gap:10px;">
#                     <strong style="color:#111827;">{html_lib.escape(os.path.basename(image_path))}</strong>
#                     <span style="color:{color}; font-weight:900;">{label}</span>
#                 </div>

#                 <p style="color:#111827; margin:10px 0 4px;">
#                     <strong style="color:#087a24;">Diagnosis:</strong> {diagnosis}
#                 </p>

#                 <p style="color:#555; font-size:13px; margin:0 0 8px;">
#                     <strong style="color:#087a24;">Mode:</strong> Gemini vision
#                 </p>

#                 <div style="color:#111827;">
#                     <strong style="color:#087a24;">Recommendations:</strong>
#                     <ul>{recs_html}</ul>
#                 </div>
#             </div>
#             """)

#         except Exception as exc:
#             reports.append(
#                 render_local_prediction_card(
#                     image_path,
#                     reason=f"Gemini failed for image #{index}: {exc}"
#                 )
#             )

#     return f"""
#     <div style="background:#fff; border:1px solid #d8e3d1; border-radius:16px;
#                 padding:22px; margin-top:20px;">
#         <h3 style="color:#087a24; margin-top:0;">🧬 Plant Scanner Diagnostics Report</h3>
#         {''.join(reports)}
#     </div>
#     """

def run_gemini_analysis(files):
    files = normalize_uploaded_files(files)

    if not files:
        return "<div style='color:#ff453a; padding:10px;'>⚠️ Upload at least one image first.</div>"

    reports = []

    for index, file_obj in enumerate(files, start=1):
        image_path = get_uploaded_file_path(file_obj)

        if not image_path or not os.path.exists(image_path):
            reports.append(
                f"""
                <div style="background:#fff; border:1px solid rgba(255,69,58,.3);
                            border-radius:14px; padding:18px; margin-top:14px;">
                    <strong style="color:#ff453a;">⚠️ Image #{index}: could not read file path.</strong>
                </div>
                """
            )
            continue

        try:
            # Use the local HuggingFace / Transformers classifier only.
            # No Gemini Vision is called here.
            reports.append(
                render_local_prediction_card(
                    image_path,
                    reason="HuggingFace/Transformers image classifier"
                )
            )

        except Exception as exc:
            reports.append(
                f"""
                <div style="background:#fff; border:1px solid rgba(255,69,58,.3);
                            border-radius:14px; padding:18px; margin-top:14px;">
                    <div style="display:flex; justify-content:space-between; flex-wrap:wrap; gap:10px;">
                        <strong style="color:#111827;">{html_lib.escape(os.path.basename(image_path))}</strong>
                        <span style="color:#ff453a; font-weight:900;">🔴 Analysis failed</span>
                    </div>

                    <p style="color:#111827; margin:10px 0 4px;">
                        <strong style="color:#087a24;">Reason:</strong>
                        {html_lib.escape(str(exc))}
                    </p>

                    <p style="color:#555; font-size:13px; margin:0;">
                        <strong style="color:#087a24;">Mode:</strong>
                        HuggingFace/Transformers image classifier
                    </p>
                </div>
                """
            )

    return f"""
    <div style="background:#fff; border:1px solid #d8e3d1; border-radius:16px;
                padding:22px; margin-top:20px;">
        <h3 style="color:#087a24; margin-top:0;">🧬 Plant Scanner Diagnostics Report</h3>

        <p style="color:#555; font-size:14px; margin-top:0;">
            The uploaded basil image is analyzed using a HuggingFace/Transformers image classifier.
            This result is an initial plant-health estimation and not a certain diagnosis.
        </p>

        {''.join(reports)}
    </div>
    """

# Sensor data microservice

from concurrent.futures import ThreadPoolExecutor, as_completed
import matplotlib.pyplot as plt
import pandas as pd
import requests
import html as html_lib
from typing import Any, Dict, List, Optional, Tuple

BASE_URL = "https://server-cloud-v645.onrender.com"
SENSORS_CONNECTED = True

SENSOR_SPECS = {
    "temperature": {"icon": "🌡️", "label": "Temperature", "unit": "°C"},
    "humidity": {"icon": "💧", "label": "Air Humidity", "unit": "%"},
    "soil": {"icon": "🌱", "label": "Soil Moisture", "unit": "%"},
}

FIREBASE_SENSOR_PATH = "sensor_readings"


def firebase_url(path: str) -> str:
    return f"{FIREBASE_DB_URL.rstrip('/')}/{path}.json"


class SensorDataService:
    """
    Sensor microservice.

    Responsibilities:
    - Read sensor history from the external Render server.
    - Normalize the server response.
    - Convert timestamps to Israel time.
    - Persist readings to Firebase.
    """

    def __init__(self, base_url: str, firebase_path: str = FIREBASE_SENSOR_PATH):
        self.base_url = base_url.rstrip("/")
        self.firebase_path = firebase_path

    def save_to_firebase(self, feed: str, df: pd.DataFrame) -> bool:
        """Save sensor rows to Firebase without creating duplicates."""
        if df.empty or "value" not in df.columns:
            return False

        payload = {}
        clean_df = df.dropna(subset=["value"]).copy()

        for i, row in clean_df.iterrows():
            created_at = row.get("created_at", "")
            if pd.notna(created_at):
                key = pd.Timestamp(created_at).strftime("%Y%m%d_%H%M%S")
                created_text = str(pd.Timestamp(created_at))
            else:
                key = f"sample_{i}"
                created_text = ""

            payload[key] = {
                "feed": feed,
                "value": float(row["value"]),
                "created_at": created_text,
            }

        if not payload:
            return False

        try:
            response = requests.patch(
                firebase_url(f"{self.firebase_path}/{feed}"),
                json=payload,
                timeout=10,
            )
            response.raise_for_status()
            return True
        except Exception as error:
            print("Could not save sensor data to Firebase:", error)
            return False

    def get_history(self, feed: str, limit: int) -> Tuple[pd.DataFrame, Dict[str, Any]]:
        """Read one sensor feed, normalize the server response, and convert to Israel Time."""
        if not SENSORS_CONNECTED:
            return pd.DataFrame(), {
                "error": "Sensors disconnected",
                "feed": feed,
            }

        try:
            safe_limit = max(1, min(int(limit), 1000))
            response = requests.get(
                f"{self.base_url}/history",
                params={"feed": feed, "limit": safe_limit},
                timeout=30,
            )
            response.raise_for_status()
            payload = response.json()
        except Exception as exc:
            return pd.DataFrame(), {"error": str(exc), "feed": feed}

        if isinstance(payload, list):
            rows = payload
        elif isinstance(payload, dict):
            rows = payload.get("data", [])
        else:
            return pd.DataFrame(), {
                "error": "The server returned an unsupported response format.",
                "feed": feed,
            }

        if not isinstance(rows, list):
            return pd.DataFrame(), {
                "error": "The server response does not contain a data list.",
                "feed": feed,
            }

        df = pd.DataFrame(rows)
        if df.empty:
            return df, payload if isinstance(payload, dict) else {"data": rows}

        if "created_at" not in df.columns:
            for candidate in ("createdAt", "timestamp", "time"):
                if candidate in df.columns:
                    df = df.rename(columns={candidate: "created_at"})
                    break

        if "value" not in df.columns:
            for candidate in ("feed_value", "sensor_value", "reading"):
                if candidate in df.columns:
                    df = df.rename(columns={candidate: "value"})
                    break

        if "created_at" in df.columns:
            df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce")

            if df["created_at"].dt.tz is None:
                df["created_at"] = df["created_at"].dt.tz_localize("UTC")

            df["created_at"] = (
                df["created_at"]
                .dt.tz_convert("Asia/Jerusalem")
                .dt.tz_localize(None)
            )

            df = df.sort_values("created_at", na_position="first").reset_index(drop=True)

        if "value" in df.columns:
            df["value"] = pd.to_numeric(df["value"], errors="coerce")

        self.save_to_firebase(feed, df)

        return df, payload if isinstance(payload, dict) else {"data": rows}


sensor_data_service = SensorDataService(BASE_URL)


def save_sensor_data_to_firebase(feed: str, df: pd.DataFrame) -> bool:
    return sensor_data_service.save_to_firebase(feed, df)


def get_sensor_data(feed: str, limit: int) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    return sensor_data_service.get_history(feed, limit)


print("SensorDataService microservice is ready.")


class BasilSimpleAgent:
    def sensor_status(self,feed: str, value: Any) -> Tuple[str, str]:
        try:
            numeric_value = float(value)
        except (TypeError, ValueError):
            return "No Data", "#aabd9e"

        if feed == "temperature":
            if numeric_value < 18:
                return "Low", "#4d8dff"
            if numeric_value <= 30:
                return "Optimal", "#30d158"
            return "High", "#ff453a"

        if feed == "humidity":
            if numeric_value < 40:
                return "Low", "#ffcc00"
            if numeric_value <= 70:
                return "Optimal", "#30d158"
            return "High", "#4d8dff"

        if feed == "soil":
            if numeric_value < 25:
                return "Low", "#ff453a"
            if numeric_value <= 80:
                return "Optimal", "#30d158"
            return "Too Wet", "#ffcc00"

        return "Unknown", "#aabd9e"

    def agent_message(self, feed, status):
        messages = {
            ("temperature", "Low"): "❄️ Low temperature detected. Basil may be too cold.",
            ("temperature", "Optimal"): "✅ Temperature is within the recommended basil range.",
            ("temperature", "High"): "🔥 High temperature detected. Basil may be under heat stress.",
            ("humidity", "Low"): "💧 Low humidity detected. Basil may need a more humid environment.",
            ("humidity", "Optimal"): "✅ Air humidity is within the preferred range.",
            ("humidity", "High"): "🌫️ High humidity detected. Risk of fungal disease may increase.",
            ("soil", "Low"): "🚿 Low soil moisture detected. Basil may need watering.",
            ("soil", "Optimal"): "✅ Soil moisture is within the preferred range.",
            ("soil", "Too Wet"): "⚠️ Soil moisture is too high. Avoid over-watering.",
        }
        return messages.get((feed, status), f"Status: {status}")

basil_agent = BasilSimpleAgent()

def check_manager_alerts(df: pd.DataFrame, feed: str) -> str:
    if df.empty or "value" not in df.columns:
        return "⚠️ No valid reading is available."

    values = df["value"].dropna()
    if values.empty:
        return "⚠️ No numeric reading is available."

    latest = float(values.iloc[-1])
    status, _ =  basil_agent.sensor_status(feed, latest)

    return basil_agent.agent_message(feed, status)


def _latest_numeric_value(df: pd.DataFrame) -> Optional[float]:
    if df.empty or "value" not in df.columns:
        return None
    values = df["value"].dropna()
    if values.empty:
        return None
    return round(float(values.iloc[-1]), 2)


def get_latest_value_for_card(feed: str, default_value: Any = "--"):
    df, _ = get_sensor_data(feed, 10)
    value = _latest_numeric_value(df)
    return value if value is not None else default_value


def get_sensor_snapshot(limit: int = 10) -> Dict[str, Dict[str, Any]]:
    snapshot: Dict[str, Dict[str, Any]] = {
        feed: {"value": None, "error": None}
        for feed in SENSOR_SPECS
    }

    with ThreadPoolExecutor(max_workers=len(SENSOR_SPECS)) as executor:
        future_to_feed = {
            executor.submit(get_sensor_data, feed, limit): feed
            for feed in SENSOR_SPECS
        }

        for future in as_completed(future_to_feed):
            feed = future_to_feed[future]
            try:
                df, raw_data = future.result()
                value = _latest_numeric_value(df)
                error = None
                if value is None:
                    if isinstance(raw_data, dict):
                        error = raw_data.get("error") or "No readings returned by the server."
                    else:
                        error = "No readings returned by the server."
                snapshot[feed] = {"value": value, "error": error}
            except Exception as exc:
                snapshot[feed] = {"value": None, "error": str(exc)}

    return snapshot


def _placeholder_snapshot() -> Dict[str, Dict[str, Any]]:
    return {
        feed: {"value": None, "error": None}
        for feed in SENSOR_SPECS
    }


def _format_sensor_value(value: Optional[float], unit: str) -> str:
    return "No data" if value is None else f"{value:g}{unit}"


def update_live_sensor_screen(feed: str, limit: int, date_filter: str = ""):
    fetch_limit = 500 if date_filter else limit
    df, raw_data = get_sensor_data(feed, fetch_limit)

    if df.empty or "value" not in df.columns or df["value"].dropna().empty:
        raw_error = raw_data.get("error", raw_data) if isinstance(raw_data, dict) else raw_data
        error_message = html_lib.escape(str(raw_error))
        return (
            f"<div class='page-card' style='color:#ff453a;'>Sensor error: {error_message}</div>",
            None,
            df,
        )

    # Apply Date Filter if provided (Now natively matches Israel Time)
    if "created_at" in df.columns and date_filter:
        df = df[df["created_at"].dt.strftime("%Y-%m-%d") == date_filter.strip()]

    df = df.tail(int(limit))

    if df.empty:
        return (
            f"<div class='page-card' style='color:#ffcc00;'>No data found for the selected date.</div>",
            None,
            df,
        )

    latest_value = float(df["value"].dropna().iloc[-1])
    manager_alert = check_manager_alerts(df, feed)

    html_result = f"""
    <div class="main-wrapper">
        <div class="page-card">
            <div class="page-title">{feed.title()} history</div>
            <div class="page-subtitle">Latest: {latest_value:.2f} · Samples displayed: {len(df)}</div>
            <div style="margin-top:18px; padding:14px; background:#f8fbf5;
                        border:1px solid rgba(126,214,98,.2); border-radius:12px;">
                {manager_alert}
            </div>
        </div>
    </div>
    """

    fig, ax = plt.subplots(figsize=(10, 4))
    if "created_at" in df.columns and df["created_at"].notna().any():
        ax.plot(df["created_at"], df["value"], marker="o")
        ax.set_xlabel("Time (Israel)")
    else:
        ax.plot(df.index, df["value"], marker="o")
        ax.set_xlabel("Sample")
    ax.set_ylabel("Value")
    ax.set_title(f"{feed.title()} history")
    plt.xticks(rotation=30)
    plt.tight_layout()

    df_table = df.sort_values("created_at", ascending=False) if "created_at" in df.columns else df.iloc[::-1]

    return html_result, fig, df_table


def make_sensor_cards_html(fetch_live: bool = True):
    snapshot = get_sensor_snapshot() if fetch_live else _placeholder_snapshot()
    cards = []
    for feed, spec in SENSOR_SPECS.items():
        value = snapshot[feed]["value"]
        status, color =  basil_agent.sensor_status(feed, value)
        value_text = _format_sensor_value(value, spec["unit"])
        message = basil_agent.agent_message(feed, status)
        error = snapshot[feed]["error"]
        error_html = (
            f"<div style='color:#ff9f0a; font-size:11px; margin-top:6px;'>"
            f"{html_lib.escape(str(error))}</div>"
            if error else ""
        )
        cards.append(f"""
        <div class="page-card" style="text-align:center;">
            <div style="font-size:30px;">{spec['icon']}</div>
            <div style="font-size:28px; font-weight:900; color:var(--text);">{value_text}</div>
            <div style="font-size:14px; font-weight:800; color:var(--text);">{spec['label']}</div>
            <div style="color:{color}; font-weight:800;">{status}</div>
            <div class="agent-message ">{message }</div>

            {error_html}
        </div>
        """)

    valid_count = sum(item["value"] is not None for item in snapshot.values())
    connection_text = "Live data" if valid_count == 3 else f"{valid_count}/3 feeds available"
    connection_color = "#30d158" if valid_count == 3 else "#ffcc00"

    return f"""
    <div class="main-wrapper">
        <div class="page-card" style="margin-bottom:24px;">
            <div style="display:flex; justify-content:space-between; align-items:center; gap:14px;">
                <div>
                    <div class="page-title">Live Telemetry</div>
                    <div class="page-subtitle">Readings from the Render sensor service</div>
                </div>
                <div style="color:{connection_color}; font-weight:800;">● {connection_text}</div>
            </div>
        </div>
        <div style="display:grid; grid-template-columns:repeat(3,1fr); gap:18px;">
            {''.join(cards)}
        </div>
    </div>
    """


def make_dashboard_sensor_summary_html(fetch_live: bool = True):
    snapshot = get_sensor_snapshot() if fetch_live else _placeholder_snapshot()

    cards = []
    for feed, spec in SENSOR_SPECS.items():
        value = snapshot[feed]["value"]
        status, color =  basil_agent.sensor_status(feed, value)
        value_text = _format_sensor_value(value, spec["unit"])
        error = snapshot[feed]["error"]
        details = html_lib.escape(str(error)) if error else status

        cards.append(f"""
        <div class="info-mini-card" style="min-height:105px;">
            <div style="font-size:24px; margin-bottom:6px;">{spec['icon']}</div>
            <div style="color:var(--muted); font-size:12px; font-weight:800;">{spec['label'].upper()}</div>
            <div style="color:var(--text); font-size:23px; font-weight:900; margin-top:4px;">{value_text}</div>
            <div style="color:{color}; font-size:12px; font-weight:800; margin-top:5px;">{details}</div>
        </div>
        """)

    valid_count = sum(item["value"] is not None for item in snapshot.values())
    if valid_count == 3:
        connection_text = "● All sensor feeds are available"
        connection_color = "#30d158"
    elif valid_count:
        connection_text = f"● Partial data: {valid_count}/3 feeds are available"
        connection_color = "#ffcc00"
    else:
        connection_text = "● Sensor service did not return data"
        connection_color = "#ff453a"

    # Enforce Israel Time for the Last Refresh timestamp
    updated_at = pd.Timestamp.now(tz='Asia/Jerusalem').strftime("%H:%M:%S")

    return f"""
    <div class="main-wrapper">
        <div class="page-card" style="margin:24px 0;">
            <div style="display:flex; justify-content:space-between; align-items:center; gap:12px; flex-wrap:wrap;">
                <div>
                    <div style="font-size:20px; font-weight:900; color:var(--text);">Sensor Summary</div>
                    <div style="color:var(--muted); font-size:12px; margin-top:4px;">Last refresh: {updated_at} (IL)</div>
                </div>
                <div style="color:{connection_color}; font-weight:800; font-size:13px;">{connection_text}</div>
            </div>
            <div style="display:grid; grid-template-columns:repeat(3,1fr); gap:14px; margin-top:18px;">
                {''.join(cards)}
            </div>
        </div>
    </div>
    """


def refresh_dashboard_sensor_summary():
    return make_dashboard_sensor_summary_html(fetch_live=True)


def refresh_live_sensor_cards():
    return make_sensor_cards_html(fetch_live=True)


def show_sensor_graph(feed: str, limit: int, date_filter: str = ""):
    if not feed:
        feed = "temperature"
    html_result, fig, df = update_live_sensor_screen(feed, limit, date_filter)
    return feed, refresh_live_sensor_cards(), html_result, fig, df

"""##4. UI Components"""

# ==========================================
# CELL 5: Frontend UI & Application Launch
# ==========================================

GLOBAL_CSS = """
/* My Basil Garden - stable light UI for Gradio/Colab */
:root,
gradio-app,
.gradio-container {
    --bg: #f6f8f4 !important;
    --card: #ffffff !important;
    --card-soft: #fbfdf8 !important;
    --text: #1f2937 !important;
    --text-soft: #374151 !important;
    --muted: #647067 !important;
    --green: #087a24 !important;
    --green-2: #2f7d32 !important;
    --green-soft: #eef7ea !important;
    --orange: #f45105 !important;
    --orange-dark: #d84300 !important;
    --border: #d8e3d1 !important;
    --shadow: 0 8px 22px rgba(31, 41, 55, 0.08) !important;
    --blue: #2563eb !important;
    color-scheme: light !important;
}

html,
body,
gradio-app {
    background: var(--bg) !important;
    color: var(--text) !important;
    color-scheme: light !important;
    overflow-x: hidden !important;
}

.gradio-container {
    max-width: 1150px !important;
    width: 100% !important;
    margin: 0 auto !important;
    padding: 18px !important;
    background: var(--bg) !important;
    color: var(--text) !important;
    font-family: Arial, Helvetica, sans-serif !important;
}

.gradio-container * {
    font-family: Arial, Helvetica, sans-serif !important;
    box-sizing: border-box !important;
}

.main-wrapper {
    width: 100% !important;
    max-width: 1080px !important;
    margin: 0 auto !important;
    padding: 8px !important;
    overflow-x: visible !important;
}

/* Header */
.app-header {
    background: var(--card) !important;
    border: 1px solid var(--border) !important;
    border-radius: 16px !important;
    padding: 18px 22px !important;
    margin: 8px 0 18px 0 !important;
    box-shadow: var(--shadow) !important;
}
.logo-box {
    display: flex !important;
    align-items: center !important;
    gap: 12px !important;
}
.logo-icon {
    width: 44px !important;
    height: 44px !important;
    border-radius: 12px !important;
    background: var(--green-soft) !important;
    display: flex !important;
    align-items: center !important;
    justify-content: center !important;
    font-size: 24px !important;
}
.logo-title {
    color: var(--green) !important;
    font-size: 24px !important;
    font-weight: 900 !important;
    line-height: 1.15 !important;
}
.logo-subtitle {
    color: var(--text-soft) !important;
    font-size: 14px !important;
    line-height: 1.25 !important;
}

/* Tabs */
.tab-nav,
[role="tablist"] {
    background: transparent !important;
    border-bottom: 1px solid #2f3a2f !important;
    overflow-x: auto !important;
    white-space: nowrap !important;
}
.tab-nav button,
button[role="tab"] {
    color: var(--text-soft) !important;
    background: transparent !important;
    border: 0 !important;
    border-radius: 10px 10px 0 0 !important;
    font-weight: 800 !important;
    padding: 9px 12px !important;
    min-height: 36px !important;
}
.tab-nav button.selected,
button[role="tab"][aria-selected="true"] {
    color: var(--green) !important;
    background: var(--green-soft) !important;
    border: 1px solid #111827 !important;
    border-bottom: 2px solid var(--orange) !important;
}

/* Cards */
.page-card,
.metric-card,
.screen-card,
.summary-card,
.upload-zone,
.reminder-left-card,
.reminder-right-card,
.manager-reminder-header,
.manager-reminder-form,
.task-card-pretty,
.reminder-task-card,
.info-mini-card,
.search-panel,
.live-card,
.sensor-card {
    background: var(--card) !important;
    color: var(--text) !important;
    border: 1px solid var(--border) !important;
    border-radius: 16px !important;
    padding: 18px !important;
    box-shadow: var(--shadow) !important;
}

.page-title,
h1,
h2,
h3,
strong {
    color: var(--green) !important;
    font-weight: 900 !important;
}
.page-subtitle,
.subtitle,
p,
label,
span,
div {
    color: var(--text) !important;
}

/* Fix old inline dark-theme colors that were left inside HTML strings */
.gradio-container [style*="color:white"],
.gradio-container [style*="color: white"],
.gradio-container [style*="color:#ffffff"],
.gradio-container [style*="color: #ffffff"],
.gradio-container [style*="color:#e8f0e4"],
.gradio-container [style*="color: #e8f0e4"] {
    color: var(--text) !important;
}
.gradio-container [style*="color:#aabd9e"],
.gradio-container [style*="color: #aabd9e"],
.gradio-container [style*="color:#9cb896"],
.gradio-container [style*="color: #9cb896"],
.gradio-container [style*="color:#8ea88a"],
.gradio-container [style*="color: #8ea88a"] {
    color: var(--muted) !important;
}
.gradio-container [style*="background:rgba(0,0,0"],
.gradio-container [style*="background: rgba(0,0,0"],
.gradio-container [style*="background:linear-gradient"],
.gradio-container [style*="background: linear-gradient"] {
    background: var(--card-soft) !important;
    border-color: var(--border) !important;
}

/* Gradio containers, inputs, file upload, gallery */
.block,
.form,
.panel,
.wrap,
.input-container,
.output-container,
.gr-box,
.gr-form,
.gr-block,
.gr-group,
.gr-column,
.gr-row,
fieldset,
textarea,
input,
select,
.gr-textbox textarea,
.gr-number input,
.gr-dropdown,
.gr-file,
.gr-image,
.file-preview,
.file-upload,
.upload-container,
.dropzone,
label[data-testid],
[data-testid="file-upload"],
[data-testid="block-label"],
[data-testid="image"] {
    background: var(--card) !important;
    color: var(--text) !important;
    border-color: var(--border) !important;
}

textarea,
input,
select,
.gr-textbox textarea,
.gr-number input {
    border: 1px solid var(--border) !important;
    border-radius: 8px !important;
    padding: 10px !important;
}
textarea::placeholder,
input::placeholder {
    color: #6b7280 !important;
    opacity: 1 !important;
}

/* Buttons */
button {
    border-radius: 10px !important;
    font-weight: 800 !important;
    min-height: 40px !important;
}
button.primary,
button[variant="primary"],
#academic-search-button,
.custom-run-btn {
    background: var(--orange) !important;
    color: #ffffff !important;
    border: 1px solid var(--orange) !important;
}
button.secondary,
button[variant="secondary"] {
    background: #555761 !important;
    color: #ffffff !important;
    border: 1px solid #555761 !important;
}

/* Tables and dataframes */
table {
    width: 100% !important;
    border-collapse: collapse !important;
    background: var(--card) !important;
    color: var(--text) !important;
}
th {
    background: var(--green-soft) !important;
    color: var(--green) !important;
    font-weight: 900 !important;
    padding: 10px !important;
    border: 1px solid var(--border) !important;
}
td {
    background: var(--card) !important;
    color: var(--text) !important;
    padding: 10px !important;
    border: 1px solid var(--border) !important;
}

.gr-dataframe,
.gr-dataframe *,
.ag-theme-quartz,
.ag-theme-quartz *,
.ag-root-wrapper,
.ag-root,
.ag-header,
.ag-header-row,
.ag-header-cell,
.ag-cell,
.ag-row,
.ag-center-cols-container,
.ag-body-viewport,
.ag-center-cols-viewport {
    background: var(--card) !important;
    color: var(--text) !important;
    border-color: var(--border) !important;
}
.ag-header,
.ag-header-cell {
    background: var(--green-soft) !important;
}
.ag-header-cell-text {
    color: var(--green) !important;
    font-weight: 900 !important;
}
.ag-cell-value,
.ag-cell {
    color: var(--text) !important;
}

/* Links and markdown/code outputs */
a {
    color: var(--blue) !important;
    font-weight: 800 !important;
}
pre,
code,
.output-text,
.output-text *,
.gr-textbox textarea {
    background: var(--card) !important;
    color: var(--text) !important;
}

/* Make all old fixed 3-column grids responsive so Colab scrolling/zoom does not break layout */
.gradio-container [style*="display:grid"] {
    width: 100% !important;
}
.gradio-container [style*="grid-template-columns:repeat(3"],
.gradio-container [style*="grid-template-columns: repeat(3"] {
    grid-template-columns: repeat(auto-fit, minmax(240px, 1fr)) !important;
}


/* HARD FIX: Academic Search result text and selected text */
.results-container,
.results-container *,
.research-summary-card,
.research-summary-card *,
.research-summary-body,
.research-summary-body *,
.research-paper-card,
.research-paper-card * {
    color: #111827 !important;
    -webkit-text-fill-color: #111827 !important;
    opacity: 1 !important;
}

.results-container h1,
.results-container h2,
.results-container h3,
.results-container h4,
.results-container strong,
.research-summary-card h1,
.research-summary-card h2,
.research-summary-card h3,
.research-summary-card h4,
.research-summary-card strong,
.research-paper-card strong {
    color: #087a24 !important;
    -webkit-text-fill-color: #087a24 !important;
    font-weight: 900 !important;
}

.results-container a,
.research-summary-card a,
.research-paper-card a {
    color: #0645ad !important;
    -webkit-text-fill-color: #0645ad !important;
    font-weight: 800 !important;
    text-decoration: underline !important;
}

.results-container p,
.results-container li,
.results-container ul,
.results-container ol,
.research-summary-body p,
.research-summary-body li,
.research-summary-body ul,
.research-summary-body ol {
    color: #111827 !important;
    -webkit-text-fill-color: #111827 !important;
    font-size: 15px !important;
    line-height: 1.7 !important;
}

/* Browser text selection: prevents the selected sentence from becoming white-on-blue */
html ::selection,
body ::selection,
gr-app ::selection,
gradio-app ::selection,
.gradio-container ::selection,
.gradio-container *::selection,
.results-container ::selection,
.results-container *::selection,
.research-summary-body ::selection,
.research-summary-body *::selection {
    background: #dff5df !important;
    color: #111827 !important;
    -webkit-text-fill-color: #111827 !important;
}

html ::-moz-selection,
body ::-moz-selection,
.gradio-container ::-moz-selection,
.gradio-container *::-moz-selection,
.results-container ::-moz-selection,
.results-container *::-moz-selection,
.research-summary-body ::-moz-selection,
.research-summary-body *::-moz-selection {
    background: #dff5df !important;
    color: #111827 !important;
}

footer {
    display: none !important;
}

@media (max-width: 700px) {
    .gradio-container { padding: 10px !important; }
    .logo-title { font-size: 20px !important; }
    .main-wrapper { padding: 4px !important; }
    .page-card,
    .screen-card,
    .summary-card,
    .upload-zone,
    .info-mini-card {
        padding: 14px !important;
    }
}

.agent-message {
    margin-top:8px;
    font-size: 13px;
    color: #e5e7eb;
    line-height: 1.4;
}

/* Game tab + dashboard game cards */
.game-tab-shell,
.dashboard-game-shell {
    width: 100% !important;
    max-width: 1080px !important;
    margin: 0 auto !important;
}
.game-rule-grid {
    display: grid !important;
    grid-template-columns: repeat(auto-fit, minmax(220px, 1fr)) !important;
    gap: 14px !important;
    margin-top: 16px !important;
}
.game-pill {
    display: inline-block !important;
    padding: 5px 10px !important;
    border-radius: 999px !important;
    background: var(--green-soft) !important;
    color: var(--green) !important;
    font-size: 12px !important;
    font-weight: 900 !important;
    margin: 3px 5px 3px 0 !important;
}
.dashboard-split-shell {
    gap: 18px !important;
    align-items: flex-start !important;
    margin-top: 18px !important;
}
.dashboard-split-shell > .gr-column {
    min-width: 0 !important;
}
.reminder-right-card,
.reminder-left-card {
    width: 100% !important;
    height: auto !important;
}
.agent-message {
    color: var(--muted) !important;
}


/* RAG chatbot bubble styling */
#rag-chatbot-ui,
.rag-chatbot-ui {
    background: #f8fafc !important;
    border: 1px solid var(--border) !important;
    border-radius: 18px !important;
    padding: 10px !important;
}

/* Prevent the message row / bubble from collapsing to a near-zero
   intrinsic width, which forces short replies like "hey" to wrap one
   character per line. Only the outer row is allowed to size itself to
   its content (max-content); every nested wrapper below it simply fills
   100% of that row so the browser doesn't have to resolve conflicting
   auto-sizing at multiple nested levels at once. */
#rag-chatbot-ui .message-row,
.rag-chatbot-ui .message-row {
    width: max-content !important;
    max-width: 85% !important;
}
#rag-chatbot-ui .message-row.user-row,
.rag-chatbot-ui .message-row.user-row {
    margin-left: auto !important;
}
#rag-chatbot-ui .message-row.bot-row,
.rag-chatbot-ui .message-row.bot-row {
    margin-right: auto !important;
}
#rag-chatbot-ui .message-row .flex-wrap,
.rag-chatbot-ui .message-row .flex-wrap,
#rag-chatbot-ui .message,
.rag-chatbot-ui .message,
#rag-chatbot-ui .message.panel-full-width,
.rag-chatbot-ui .message.panel-full-width {
    width: 100% !important;
    max-width: 100% !important;
    border-radius: 18px !important;
    padding: 12px 14px !important;
    line-height: 1.55 !important;
    box-shadow: 0 8px 22px rgba(15, 23, 42, 0.08) !important;
}
#rag-chatbot-ui .message-row .flex-wrap,
.rag-chatbot-ui .message-row .flex-wrap,
#rag-chatbot-ui .message.panel-full-width,
.rag-chatbot-ui .message.panel-full-width {
    box-shadow: none !important;
    padding: 0 !important;
}

#rag-chatbot-ui .message.user,
.rag-chatbot-ui .message.user {
    background: #dcfce7 !important;
    border: 1px solid #bbf7d0 !important;
    color: #14532d !important;
    margin-left: auto !important;
}

#rag-chatbot-ui .message.bot,
.rag-chatbot-ui .message.bot {
    background: #ffffff !important;
    border: 1px solid #e5e7eb !important;
    color: #111827 !important;
    margin-right: auto !important;
}

#rag-chatbot-ui .message p,
.rag-chatbot-ui .message p,
#rag-chatbot-ui .message li,
.rag-chatbot-ui .message li {
    color: inherit !important;
}

#rag-chatbot-ui .avatar-container,
.rag-chatbot-ui .avatar-container {
    filter: drop-shadow(0 4px 10px rgba(15, 23, 42, 0.12)) !important;
}
/* Chatbot text visibility fix */
.chatbot-readable,
.chatbot-readable * {
    color: #111827 !important;
}

.chatbot-readable p,
.chatbot-readable span,
.chatbot-readable div,
.chatbot-readable li,
.chatbot-readable strong,
.chatbot-readable em {
    color: #111827 !important;
}

/* Bot and user bubbles */
.chatbot-readable .message,
.chatbot-readable .bubble,
.chatbot-readable .prose {
    color: #111827 !important;
}

/* Markdown text inside bot answer */
.chatbot-readable .prose *,
.chatbot-readable markdown,
.chatbot-readable markdown * {
    color: #111827 !important;
}
.chatbot-readable [data-testid="bot"],
.chatbot-readable [data-testid="bot"] * ,
.chatbot-readable [data-testid="user"],
.chatbot-readable [data-testid="user"] * {
    color: #111827 !important;
}
/* ✅ FIX: Plant Scanner output text always dark */
#component-plant-output,
#component-plant-output *,
.plant-scan-result,
.plant-scan-result * {
    color: #111827 !important;
    -webkit-text-fill-color: #111827 !important;
}

.small-image-gallery,
.small-image-gallery *,
[data-testid="gallery"],
[data-testid="gallery"] * {
    background: #ffffff !important;
    background-color: #ffffff !important;
}
css.small-image-gallery,
[data-testid="gallery"] {
    background: #ffffff !important;
}
[data-testid="gallery"] img {
    object-fit: contain !important;
}

.small-image-gallery {
    background: #ffffff !important;
}
.small-image-gallery img,
[data-testid="gallery"] img {
    object-fit: contain !important;
    border-radius: 8px !important;
}
[data-testid="gallery"],
[data-testid="gallery"] > div {
    background: #ffffff !important;
    background-color: #ffffff !important;
}
/* File upload clean style */
.custom-file {
    background: #ffffff !important;
    border: 2px dashed #d8e3d1 !important;
    border-radius: 12px !important;
    min-height: 80px !important;
}
.custom-file * {
    color: #1f2937 !important;
}

.custom-file [data-testid="file-preview"] {
    background: #ffffff !important;
}
.custom-file .file-preview,
.custom-file [data-testid="file-preview"] {
    display: none !important;
}
"""

def show_dashboard():
    return (
        gr.update(visible=True),
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=False),
        refresh_dashboard_sensor_summary(),
    )


def show_upload():
    return (
        gr.update(visible=False),
        gr.update(visible=True),
        gr.update(visible=False),
        gr.update(visible=False),
    )


def show_research():
    return (
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=True),
        gr.update(visible=False),
    )


def show_live():
    return (
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=True),
        refresh_live_sensor_cards(),
    )

###################
# dashboard_html  #
###################
dashboard_html = """
<div class="main-wrapper">

        <div style="display:grid; grid-template-columns: repeat(3, 1fr); gap:18px; margin-bottom:24px;">

        <div class="page-card">
            <div style="font-size:26px; margin-bottom:8px;">📷</div>
            <div style="color:var(--muted); font-size:12px; font-weight:800;">PLANT SCANNER</div>
            <div style="color:var(--text); font-size:22px; font-weight:900; margin-top:6px;">Ready</div>
            <div style="color:#30d158; font-size:12px; margin-top:8px;">Image upload available</div>
        </div>

        <div class="page-card">
            <div style="font-size:26px; margin-bottom:8px;">📚</div>
            <div style="color:var(--muted); font-size:12px; font-weight:800;">ACADEMIC SEARCH</div>
            <div style="color:var(--text); font-size:22px; font-weight:900; margin-top:6px;">Active</div>
            <div style="color:#30d158; font-size:12px; margin-top:8px;">Boolean search enabled</div>
        </div>

        <div class="page-card">
            <div style="font-size:26px; margin-bottom:8px;">📡</div>
            <div style="color:var(--muted); font-size:12px; font-weight:800;">LIVE TELEMETRY</div>
            <div style="color:var(--text); font-size:22px; font-weight:900; margin-top:6px;">Monitoring</div>
            <div style="color:var(--muted); font-size:12px; margin-top:8px;">Live status is shown below</div>
        </div>

    </div>

</div>
"""

# ==========================================
# MANAGER SETUP / TASKS COMPONENT + FIREBASE
# ==========================================

import time
import html as html_lib
import requests

FIREBASE_MANAGER_TASKS_PATH = "manager_tasks"


def firebase_tasks_url():
    return "https://cloudroot-9da6c-default-rtdb.europe-west1.firebasedatabase.app/manager_tasks.json"


def load_manager_tasks():
    try:
        response = requests.get(firebase_tasks_url(), timeout=10)
        response.raise_for_status()
        data = response.json()

        if not data:
            return []

        if isinstance(data, dict):
            data = list(data.values())

        return [
            {
                "id": task.get("id", int(time.time() * 1000)),
                "task": str(task.get("task", "")),
                "cost": float(task.get("cost", 0)),
            }
            for task in data
            if isinstance(task, dict) and task.get("task")
        ]

    except Exception as error:
        print("Could not load manager tasks from Firebase:", error)
        return []


def save_manager_tasks(tasks):
    try:
        response = requests.put(firebase_tasks_url(), json=list(tasks or []), timeout=10)
        response.raise_for_status()
        return True

    except Exception as error:
        print("Could not save manager tasks to Firebase:", error)
        return False


def render_task_card(task_data, index):
    task_text = html_lib.escape(str(task_data.get("task", "")))
    task_cost = html_lib.escape(str(task_data.get("cost", "")))

    return f"""
    <div class="reminder-task-card">
        <div style="display:flex; justify-content:space-between; align-items:flex-start; gap:14px;">
            <div>
                <div style="color:#9fc99a; font-size:11px; font-weight:850; text-transform:uppercase; letter-spacing:0.08em; margin-bottom:6px;">
                    Reminder #{index + 1}
                </div>
                <div style="color:var(--text); font-size:18px; font-weight:850; line-height:1.35;">
                    {task_text}
                </div>
            </div>

            <div style="
                min-width:90px;
                text-align:right;
                background:#f8fbf5;
                border: 1px solid var(--border);
                border-radius: 14px;
                padding: 10px 12px;
            ">
                <div style="color:var(--muted); font-size:10px; font-weight:850; text-transform:uppercase;">
                    Unit cost
                </div>
                <div style="color:#7edf62; font-size:18px; font-weight:900; margin-top:4px;">
                    ₪{task_cost}
                </div>
            </div>
        </div>
    </div>
    """


def _normalize_game_outputs(xp_outputs):
    if xp_outputs is None:
        return []
    if isinstance(xp_outputs, (list, tuple)):
        return list(xp_outputs)
    return [xp_outputs]


def _repeat_game_html(html, xp_outputs):
    return [html for _ in _normalize_game_outputs(xp_outputs)]


def build_manager_setup_section(xp_outputs=None):
    game_outputs = _normalize_game_outputs(xp_outputs)
    tasks_state = gr.State(load_manager_tasks())

    with gr.Row(elem_classes=["main-wrapper", "dashboard-split-shell"]):
        with gr.Column(scale=2, min_width=420):
            with gr.Group(elem_classes=["reminder-left-card"]):
                gr.HTML("""
                <div style="font-size:22px; font-weight:900; color:var(--green);">
                    📝 Plant Care Tasks
                </div>
                <div style="font-size:13px; color:var(--muted); margin-bottom:12px;">
                    Add preventive care actions. Each saved task gives care XP and helps complete the daily care mission.
                </div>
                """)

                task_input = gr.Textbox(label="Task", placeholder="e.g., Water basil plant")
                unit_cost_input = gr.Number(label="Unit Cost (₪)", value=0)
                add_task_btn = gr.Button("➕ Add Task", variant="primary")
                manager_message = gr.HTML("")

                def add_new_task(text, cost, current_tasks):
                    current_tasks = list(current_tasks or [])

                    if not text or not text.strip():
                        data = load_game_data()
                        dashboard_html, game_html = render_game_outputs(data["xp"], data["missions"])
                        msg = "<div style='color:#d84300; font-weight:800;'>Please enter a task.</div>"
                        return (current_tasks, "", 0, msg, dashboard_html, game_html)

                    try:
                        cost = float(cost)
                    except Exception:
                        cost = 0

                    new_tasks = current_tasks + [{
                        "id": int(time.time() * 1000),
                        "task": text.strip(),
                        "cost": cost,
                    }]

                    saved = save_manager_tasks(new_tasks)

                    if not saved:
                        data = load_game_data()
                        dashboard_html, game_html = render_game_outputs(data["xp"], data["missions"])
                        msg = (
                            "<div style='color:#d84300; font-weight:800;'>"
                            "Firebase update failed. The task was not saved. "
                            "Check the Firebase URL / database rules, and look at the Colab output for the exact error."
                            "</div>"
                        )
                        return (current_tasks, text, cost, msg, dashboard_html, game_html)

                    _, dashboard_html, game_html = process_game_action("care")
                    msg = "<div style='color:var(--green); font-weight:800;'>Task saved to Firebase.</div>"
                    return (new_tasks, "", 0, msg, dashboard_html, game_html)

                add_task_btn.click(
                    fn=add_new_task,
                    inputs=[task_input, unit_cost_input, tasks_state],
                    outputs=[tasks_state, task_input, unit_cost_input, manager_message, *game_outputs],
                )

                @gr.render(inputs=tasks_state)
                def render_dynamic_tasks(tasks_list):
                    if not tasks_list:
                        gr.HTML("""
                        <div style="color:var(--muted); text-align:center; padding:18px; margin-top:14px; border:1px dashed var(--border); border-radius:14px;">
                            No tasks yet. Add your first preventive care task above.
                        </div>
                        """)
                        return

                    for i, task_data in enumerate(tasks_list):
                        gr.HTML(render_task_card(task_data, i))
                        with gr.Row():
                            edit_btn = gr.Button("Edit")
                            delete_btn = gr.Button("Delete")

                        def delete_this_task(current_tasks, index=i):
                            current_tasks = list(current_tasks or [])
                            if index < len(current_tasks):
                                current_tasks.pop(index)
                            save_manager_tasks(current_tasks)
                            return current_tasks, "<div style='color:var(--green); font-weight:800;'>Task deleted successfully.</div>"

                        def edit_this_task(current_tasks, index=i):
                            current_tasks = list(current_tasks or [])
                            if index >= len(current_tasks):
                                return current_tasks, "", 0, ""
                            task_to_edit = current_tasks.pop(index)
                            save_manager_tasks(current_tasks)
                            return (
                                current_tasks,
                                task_to_edit.get("task", ""),
                                task_to_edit.get("cost", 0),
                                "<div style='color:#d84300; font-weight:800;'>Edit the task above and click Add Task.</div>",
                            )

                        delete_btn.click(fn=delete_this_task, inputs=[tasks_state], outputs=[tasks_state, manager_message])
                        edit_btn.click(fn=edit_this_task, inputs=[tasks_state], outputs=[tasks_state, task_input, unit_cost_input, manager_message])

        with gr.Column(scale=1, min_width=320):
            with gr.Group(elem_classes=["reminder-right-card"]):
                gr.HTML("""
                <div style="margin-bottom:18px;">
                    <div style="font-size:22px; font-weight:900; color:var(--green); margin-bottom:6px;">
                        About This Dashboard
                    </div>
                    <div style="font-size:14px; color:var(--muted); line-height:1.6;">
                        This area keeps the plant-care tasks organized and connected to the Health Garden Race.
                        Tasks are stored in Firebase and count as preventive actions in the game.
                    </div>
                </div>

                <div class="info-mini-card">
                    <div style="font-size:15px; font-weight:900; color:var(--text); margin-bottom:5px;">🌱 What can you track?</div>
                    <div style="font-size:13px; color:var(--muted);">Temperature, humidity, soil moisture, image uploads, searches, and care tasks.</div>
                </div>

                <div class="info-mini-card" style="margin-top:12px;">
                    <div style="font-size:15px; font-weight:900; color:var(--text); margin-bottom:5px;">📌 Task examples</div>
                    <div style="font-size:13px; color:var(--muted);">Water basil, inspect leaves, clean sensors, move the plant, or check humidity.</div>
                </div>

                <div class="info-mini-card" style="margin-top:12px;">
                    <div style="font-size:15px; font-weight:900; color:var(--text); margin-bottom:5px;">🏆 Game connection</div>
                    <div style="font-size:13px; color:var(--muted);">Every useful care task gives XP and helps complete daily plant-care missions.</div>
                </div>
                """)

# ==========================================
# GAMIFICATION + ACADEMIC RAG CHATBOT
# ==========================================

from datetime import datetime
try:
    from zoneinfo import ZoneInfo
    ISRAEL_TZ = ZoneInfo("Asia/Jerusalem")
except Exception:
    ISRAEL_TZ = None

FIREBASE_GAME_URL = f"{FIREBASE_DB_URL.rstrip('/')}/basil_gamification.json"

GAME_RULES = {
    "sensor": {"xp": 10, "cooldown": 300},
    "search": {"xp": 20, "cooldown": 120},
    "scan": {"xp": 25, "cooldown": 300},
    "care": {"xp": 15, "cooldown": 60},
    "daily_search_target": 2,
    "daily_care_target": 1,
    "weekly_sensor_target": 5,
    "weekly_scan_target": 2,
    "daily_bonus": 50,
    "weekly_bonus": 100,
}

DEFAULT_MISSIONS = [
    "Check temperature, humidity, and soil moisture.",
    "Search the academic index for basil disease prevention.",
    "Add one preventive care task, such as watering or checking leaves.",
    "Upload a plant image when you see a suspicious symptom.",
]


def gemini_connected() -> bool:
    return bool(GEMINI_AVAILABLE and GEMINI_API_KEY and genai is not None)


def get_israel_time():
    return datetime.now(ISRAEL_TZ) if ISRAEL_TZ else datetime.now()


def default_game_data():
    return {
        "xp": 0,
        "last_actions": {},
        "missions": {
            "daily_date": "",
            "weekly_week": "",
            "daily_searches": 0,
            "daily_care": 0,
            "weekly_sensors": 0,
            "weekly_scans": 0,
        },
    }



def ensure_game_data_shape(data):
    """Keep Firebase game data compatible when old records are missing new fields."""
    defaults = default_game_data()

    if not isinstance(data, dict):
        return defaults

    data.setdefault("xp", defaults["xp"])

    if not isinstance(data.get("last_actions"), dict):
        data["last_actions"] = {}

    if not isinstance(data.get("missions"), dict):
        data["missions"] = {}

    for key, value in defaults["missions"].items():
        data["missions"].setdefault(key, value)

    return data


def load_game_data():
    data = default_game_data()
    try:
        response = requests.get(FIREBASE_GAME_URL, timeout=10)
        if response.status_code == 200 and isinstance(response.json(), dict):
            remote = response.json()

            # Do not replace the whole missions dictionary, because older
            # Firebase records may not contain newer keys such as ai_chat_missions.
            data["xp"] = remote.get("xp", data["xp"])
            data["last_actions"] = remote.get("last_actions", data["last_actions"])
            data["missions"].update(remote.get("missions", {}))

    except Exception as error:
        print("Could not load game data:", error)

    data = ensure_game_data_shape(data)
    reset_missions_if_needed(data)
    data = ensure_game_data_shape(data)
    return data


def save_game_data(data):
    try:
        requests.put(FIREBASE_GAME_URL, json=data, timeout=10)
    except Exception as error:
        print("Could not save game data:", error)


def reset_missions_if_needed(data):
    data = ensure_game_data_shape(data)
    today = get_israel_time().strftime("%Y-%m-%d")
    week = get_israel_time().strftime("%Y-W%U")
    missions = data["missions"]

    if missions.get("daily_date") != today:
        missions["daily_date"] = today
        missions["daily_searches"] = 0
        missions["daily_care"] = 0

    if missions.get("weekly_week") != week:
        missions["weekly_week"] = week
        missions["weekly_sensors"] = 0
        missions["weekly_scans"] = 0

    ensure_game_data_shape(data)


def get_basil_rank(xp):
    if xp < 100:
        return "🌱 Dormant Seed"
    if xp < 400:
        return "🌿 First Sprout"
    if xp < 1000:
        return "🍃 Vibrant Leaf"
    if xp < 2500:
        return "🌳 Strong Branch"
    return "👑 Basil Master"


def progress_bar(value, target, color="var(--green)"):
    percent = min(100, int((value / target) * 100)) if target else 0
    return f"""
    <div style="width:100%; height:8px; background:#eef7ea; border-radius:6px; overflow:hidden;">
        <div style="width:{percent}%; height:100%; background:{color}; border-radius:6px;"></div>
    </div>
    """


def mission_card(title, value, target, color):
    return f"""
    <div class="info-mini-card">
        <div style="display:flex; justify-content:space-between; gap:10px; font-size:12px; font-weight:900;">
            <span>{title}</span>
            <span>{value}/{target}</span>
        </div>
        <div style="margin-top:8px;">{progress_bar(value, target, color)}</div>
    </div>
    """


def render_leaderboard(xp):
    users = [("You", xp), ("Green Team", 820), ("Basil Pro", 540), ("Care Starter", 180)]
    users = sorted(users, key=lambda item: item[1], reverse=True)
    rows = ""
    for place, (name, score) in enumerate(users, start=1):
        style = "background:#eef7ea; font-weight:900;" if name == "You" else ""
        rows += f"""
        <div style="display:flex; justify-content:space-between; padding:8px 10px; border-radius:10px; {style}">
            <span>#{place} {name}</span><span>{score} XP</span>
        </div>
        """
    return f"""
    <div class="info-mini-card">
        <div style="font-weight:900; color:var(--text); margin-bottom:8px;">🏅 Garden Leaderboard</div>
        {rows}
    </div>
    """


def render_xp_dashboard(xp, missions=None, messages=None, show_leaderboard=True):
    missions = missions or default_game_data()["missions"]
    leaderboard_html = render_leaderboard(xp) if show_leaderboard else ""
    rank = get_basil_rank(xp)
    progress = min(100, int((xp % 1000) / 10)) if xp < 2500 else 100
    ai_status = "Chatbot uses Academic Search / RAG. Gemini is used only when connected." if gemini_connected() else "Chatbot uses Academic Search / RAG with local fallback because Gemini is not connected."

    msg_html = ""
    if messages:
        safe_messages = [html_lib.escape(str(msg)) for msg in messages]
        msg_html = "<div style='margin-top:14px; color:#d84300; font-weight:800;'>" + "<br>".join(safe_messages) + "</div>"

    missions_html = "".join([
        mission_card("📚 Daily academic searches", missions.get("daily_searches", 0), GAME_RULES["daily_search_target"], "#8b5cf6"),
        mission_card("📝 Daily preventive care", missions.get("daily_care", 0), GAME_RULES["daily_care_target"], "var(--green)"),
        mission_card("📡 Weekly sensor checks", missions.get("weekly_sensors", 0), GAME_RULES["weekly_sensor_target"], "var(--blue)"),
        mission_card("📷 Weekly plant scans", missions.get("weekly_scans", 0), GAME_RULES["weekly_scan_target"], "var(--orange)"),
    ])

    return f"""
    <div class="page-card dashboard-game-shell" style="margin:10px auto 20px auto;">
        <div style="display:flex; justify-content:space-between; gap:20px; flex-wrap:wrap; align-items:flex-start;">
            <div>
                <div style="color:var(--muted); font-size:12px; font-weight:900;">HEALTH GARDEN RACE</div>
                <div style="color:var(--green); font-size:26px; font-weight:900;">{rank}</div>
                <div style="color:var(--muted); font-size:13px; margin-top:6px;">{ai_status}</div>
            </div>
            <div style="text-align:right;">
                <div style="color:var(--muted); font-size:12px; font-weight:900;">TOTAL XP</div>
                <div style="color:var(--orange); font-size:30px; font-weight:900;">{xp} XP</div>
            </div>
        </div>
        <div style="margin-top:14px;">{progress_bar(progress, 100, "linear-gradient(90deg, var(--green), #7edf62)")}</div>
        <div style="display:grid; grid-template-columns:repeat(auto-fit,minmax(220px,1fr)); gap:14px; margin-top:18px;">
            {missions_html}
            {leaderboard_html}
        </div>
        {msg_html}
    </div>
    """


def render_game_explanation_html():


    return f"""
    <div class="page-card game-tab-shell" style="margin:18px auto; padding:24px;">
        <div class="page-title">🏁 Health Garden Race</div>
        <div style="color:var(--text); line-height:1.7; margin-top:8px;">
            This is the gamification of My Basil Garden. The user earns XP by doing real plant-care actions:
        </div>

        <div class="game-rule-grid">
            <div class="info-mini-card">
                <div style="font-weight:900; color:var(--green);">📡 Monitoring</div>
                <div style="font-size:13px; color:var(--muted); margin-top:6px;">Check sensor data regularly and complete the weekly sensor challenge.</div>
            </div>
            <div class="info-mini-card">
                <div style="font-weight:900; color:var(--green);">🔍 Early Detection</div>
                <div style="font-size:13px; color:var(--muted); margin-top:6px;">Use image scans and academic search to identify possible plant issues earlier.</div>
            </div>
            <div class="info-mini-card">
                <div style="font-weight:900; color:var(--green);">📝 Prevention</div>
                <div style="font-size:13px; color:var(--muted); margin-top:6px;">Add care tasks like watering, checking leaves, cleaning sensors, or improving conditions.</div>
            </div>
            <div class="info-mini-card">
                <div style="font-weight:900; color:var(--green);">🏆 Competition</div>
                <div style="font-size:13px; color:var(--muted); margin-top:6px;">Compare XP with other simulated users on the leaderboard.</div>
            </div>
        </div>

        <div style="margin-top:18px;">
            <span class="game-pill">Search: +20 XP</span>
            <span class="game-pill">Care task: +15 XP</span>
            <span class="game-pill">Sensor check: +10 XP</span>
            <span class="game-pill">Plant scan: +25 XP</span>

        </div>

    </div>
    """


def refresh_game_outputs():
    data = load_game_data()
    return render_game_outputs(data["xp"], data["missions"])


def process_game_action(action_type):
    data = load_game_data()
    reset_missions_if_needed(data)

    now = time.time()
    rule = GAME_RULES[action_type]
    last_time = data["last_actions"].get(action_type, 0)
    messages = []

    if now - last_time < rule["cooldown"]:
        wait = int(rule["cooldown"] - (now - last_time))
        messages.append(f"⏳ Wait {wait}s before earning more {action_type} XP.")
        dashboard_html, game_tab_html = render_game_outputs(data["xp"], data["missions"], messages)
        return data["xp"], dashboard_html, game_tab_html

    gained = rule["xp"]
    data["last_actions"][action_type] = now
    messages.append(f"✨ +{gained} XP for {action_type}")

    missions = data["missions"]

    if action_type == "search":
        missions["daily_searches"] += 1
        if missions["daily_searches"] == GAME_RULES["daily_search_target"]:
            gained += GAME_RULES["daily_bonus"]
            messages.append(f"🏆 Daily academic search completed! +{GAME_RULES['daily_bonus']} XP")

    if action_type == "care":
        missions["daily_care"] += 1
        if missions["daily_care"] == GAME_RULES["daily_care_target"]:
            gained += GAME_RULES["daily_bonus"]
            messages.append(f"🌿 Daily preventive care completed! +{GAME_RULES['daily_bonus']} XP")

    if action_type == "sensor":
        missions["weekly_sensors"] += 1
        if missions["weekly_sensors"] == GAME_RULES["weekly_sensor_target"]:
            gained += GAME_RULES["weekly_bonus"]
            messages.append(f"📡 Weekly sensor challenge completed! +{GAME_RULES['weekly_bonus']} XP")

    if action_type == "scan":
        missions["weekly_scans"] += 1
        if missions["weekly_scans"] == GAME_RULES["weekly_scan_target"]:
            gained += GAME_RULES["weekly_bonus"]
            messages.append(f"🔍 Weekly early detection challenge completed! +{GAME_RULES['weekly_bonus']} XP")

    data["xp"] += gained
    save_game_data(data)
    dashboard_html, game_tab_html = render_game_outputs(data["xp"], data["missions"], messages)
    return data["xp"], dashboard_html, game_tab_html
def gamified_search(query):
    data = load_game_data()
    current_html = render_xp_dashboard(data["xp"], data["missions"])

    if not query or not query.strip():
        yield "<div class='page-card'>Please enter a search query.</div>", current_html, current_html
        return

    _, dashboard_html, game_html = process_game_action("search")
    for step in advanced_html_search(query):
        yield f"<div class='results-container'>{step}</div>",dashboard_html, game_html


def gamified_image_analysis(files):
    result_html = run_gemini_analysis(files)
    if files:
        _, dashboard_html, game_html = process_game_action("scan")
    else:
        data = load_game_data()
        game_html = render_xp_dashboard(data["xp"], data["missions"])
    return result_html, game_html, game_html


def gamified_sensor_refresh():
    _, dashboard_html, game_html = process_game_action("sensor")
    return refresh_dashboard_sensor_summary(), game_html, game_html


def gamified_live_cards_refresh():
    _, dashboard_html, game_html = process_game_action("sensor")
    return refresh_live_sensor_cards(), game_html, game_html


def gamified_show_sensor_graph(feed, limit, date_filter):
    result = show_sensor_graph(feed, limit, date_filter)
    _, dashboard_html, game_html = process_game_action("sensor")
    return result[0], result[1], result[2], result[3], result[4], game_html, game_html
def render_game_outputs(xp, missions, messages=None):
    dashboard_html = render_xp_dashboard(
        xp, missions, messages, show_leaderboard=False
    )

    game_tab_html = render_xp_dashboard(
        xp, missions, messages, show_leaderboard=True
    )

    return dashboard_html, game_tab_html

print("Gamification and Academic Search are ready.")

# ==========================================
# CELL 20B: STANDALONE AI CHATBOT - NO RAG
# ==========================================
# This chatbot follows Tutorial 9 style:
# Gemini AI is used first. If Gemini is unavailable or fails,
# the chatbot returns prepared pattern-based answers.
# Important: this cell does NOT call rag_system and does NOT use Academic RAG.

import re

CHATBOT_PATTERNS = [
    {
      "keywords": ["hello", "hi", "hey", "help"],
      "response": (
          "Hi! I’m your basil care assistant 🌿 "
          "You can ask me about watering, temperature, humidity, soil moisture, "
          "leaf symptoms, plant diseases, or sensor readings. "
          "I’ll help you understand what might be happening with your basil."
      )
  },
  {
      "keywords": ["temperature", "hot", "cold", "heat"],
      "response": (
          "Basil prefers warm and stable conditions 🌡️ "
          "I recommend checking the live temperature sensor. "
          "If the temperature is too high or too low, try moving the plant to a more balanced location."
      )
  },
  {
      "keywords": ["humidity", "moisture", "soil", "dry", "wet"],
      "response": (
          "Let’s check the humidity and soil moisture readings 🌱 "
          "Basil likes soil that stays slightly moist, but not too wet. "
          "If the soil moisture is very low, the plant may need watering. "
          "If it is too high, overwatering could increase disease risk."
      )
  },
  {
      "keywords": ["water", "watering", "irrigation"],
      "response": (
          "For watering, it’s best to use the soil moisture sensor instead of guessing 💧 "
          "Water the basil when the soil starts to dry, but avoid keeping it constantly wet. "
          "Balanced watering helps prevent both dryness and root stress."
      )
  },
  {
      "keywords": ["spots", "black", "yellow", "disease", "leaves", "leaf", "mildew", "fungus", "mold"],
      "response": (
          "Leaf spots, yellow leaves, mildew, or mold-like signs may point to plant stress or disease 🍃 "
          "Try uploading a clear photo in the Plant Scanner tab for an initial check. "
          "You can also use Academic Search to find article-based information about possible basil diseases."
      )
  },
  {
      "keywords": ["sensor", "sensors", "dashboard", "telemetry", "data"],
      "response": (
          "You can check the Dashboard or Live Telemetry tab to see the latest sensor readings 📡 "
          "The temperature, humidity, and soil moisture values can help you understand the plant’s condition "
          "and decide whether it needs watering or a change in its environment."
      )
  }
]


def chatbot_pattern_fallback(message):
    """Prepared pattern fallback, like the Tutorial 9 rule-based chatbot idea."""
    msg = str(message or "").lower()
    tokens = set(re.findall(r"\b\w+\b", msg))

    for item in CHATBOT_PATTERNS:
        if any(word in tokens for word in item["keywords"]):
            return item["response"]

    return (
        "I could not answer with Gemini, so I used the prepared chatbot patterns. "
        "Try asking about basil watering, humidity, temperature, soil moisture, leaf spots, mildew, or sensors."
    )


def shorten_chat_answer(text, max_chars=900):
    text = str(text or "").strip()
    text = re.sub(r"\n{3,}", "\n\n", text)

    if len(text) <= max_chars:
        return text

    sentences = re.split(r"(?<=[.!?])\s+", text)
    short = " ".join(sentences[:4]).strip()

    if len(short) > max_chars:
        short = short[:max_chars].rsplit(" ", 1)[0] + "..."

    return short + "\n\n_(Shortened for chat.)_"


def to_chat_markdown(text):
    """Return a clean Markdown string for Gradio Chatbot."""
    text = str(text or "").strip()
    text = text.replace("•", "-")
    return text


def add_chat_suggestion(question, answer):
    question = str(question or "").lower()
    answer = str(answer or "").strip()

    if "suggestion" in answer.lower():
        return answer

    greeting_words = ["hello", "hi", "hey", "help", "thanks", "thank you", "bye",
                       "שלום", "היי", "אהלן", "תודה", "ביי", "מה נשמע", "מה קורה"]
    if any(word in question for word in greeting_words):
        return answer

    if "mildew" in question or "fungus" in question or "disease" in question or "spots" in question:
        suggestion = (
            "**Suggestion:** Check the basil leaves, reduce excess humidity, improve air flow, "
            "and compare symptoms with the Plant Scanner or Academic Search tab."
        )

    elif "water" in question or "soil" in question or "moisture" in question:
        suggestion = (
            "**Suggestion:** Check the soil moisture sensor before watering, "
            "and avoid keeping the soil too wet for a long time."
        )

    elif "temperature" in question or "hot" in question or "cold" in question:
        suggestion = (
            "**Suggestion:** Check the live temperature sensor and keep the basil in stable warm conditions."
        )

    elif "humidity" in question or "sensor" in question or "dashboard" in question:
        suggestion = (
            "**Suggestion:** Use the sensor dashboard, plant scanner, or Academic Search "
            "to compare the answer with your project data."
        )

    else:
        return answer

    return answer + "\n\n---\n" + suggestion


def normalize_chat_history(history):
    """Support both old Gradio pair history and new messages history."""
    normalized = []

    for item in history or []:
        if isinstance(item, dict) and "role" in item and "content" in item:
            normalized.append({
                "role": str(item.get("role", "assistant")),
                "content": str(item.get("content", "")),
            })

        elif isinstance(item, (list, tuple)) and len(item) >= 2:
            if item[0]:
                normalized.append({"role": "user", "content": str(item[0])})
            if item[1]:
                normalized.append({"role": "assistant", "content": str(item[1])})

    return normalized


def gemini_chatbot_answer(user_message):
    """
    AI chatbot answer using Gemini.
    If Gemini fails because of quota, timeout, or API error, return None
    so the prepared chatbot patterns can be used.
    """
    if not (
        globals().get("GEMINI_AVAILABLE")
        and globals().get("GEMINI_API_KEY")
        and globals().get("genai") is not None
    ):
        print("Gemini chatbot skipped: Gemini is not configured.")
        return None

    try:
        genai.configure(api_key=GEMINI_API_KEY)
        model = genai.GenerativeModel("gemini-2.5-flash")

        prompt = f"""
            You are a friendly AI chatbot for a student project called "My Basil Garden".

            Rules:
            - Answer in the same language as the user's question.
            - Focus on basil care, watering, humidity, temperature, soil moisture, sensors, and leaf symptoms.
            - Do not use RAG in this chatbot.
            - Do not claim that you searched academic papers.
            - If the user needs article-based evidence, tell them to use the Academic Search tab.
            - Keep the answer short, clear, and practical.

            User question:
            {user_message}
            """

        response = model.generate_content(
            prompt,
            generation_config=genai.types.GenerationConfig(
                temperature=0.3,
                max_output_tokens=300,
            ),
            request_options={
                "timeout": 10
            }
        )

        answer = getattr(response, "text", "").strip()

        if not answer:
            print("Gemini chatbot returned empty answer.")
            return None

        return shorten_chat_answer(answer)

    except Exception as exc:
        print("Gemini chatbot failed:", exc)
        return None

def standalone_ai_chatbot_stream(user_message, history):
    """
    Standalone AI chatbot.
    Gemini first, prepared patterns second.
    Uses Gradio's "messages" history format: a list of {"role": ..., "content": ...} dicts.
    """
    history = normalize_chat_history(history)

    if not user_message or not str(user_message).strip():
        yield "", history
        return

    user_message = str(user_message).strip()

    history = history + [
        {"role": "user", "content": user_message},
        {"role": "assistant", "content": "🤖 Thinking..."},
    ]
    yield "", history

    answer = gemini_chatbot_answer(user_message)

    if not answer:
        answer = chatbot_pattern_fallback(user_message)

    answer = add_chat_suggestion(user_message, answer)
    answer = to_chat_markdown(answer)

    history[-1]["content"] = answer
    yield "", history

"""## Microservices used in this project

1. **EcologicalRAG service** – retrieves relevant basil papers and generates a grounded answer or local fallback.
2. **SensorDataService** – reads telemetry from the Render sensor server, normalizes it, saves it to Firebase, and serves the Dashboard / Live Telemetry tabs.
3. **PlantImageClassifierService** – loads the trained EfficientNetV2 Keras model and classifies basil images when Gemini is unavailable or fails.

#5. Frontend Application
"""

with gr.Blocks(title="My Basil Garden", css=GLOBAL_CSS, theme=gr.themes.Soft(primary_hue="green", secondary_hue="orange", neutral_hue="slate")) as app:
    gr.HTML("""
    <div class="main-wrapper">
        <div class="app-header">
            <div class="logo-box">
                <div class="logo-icon">🌿</div>
                <div>
                    <div class="logo-title">My Basil Garden</div>
                    <div class="logo-subtitle">Intelligent Plant Monitoring</div>
                    <div class="logo-subtitle">Cloud Root Project</div>
                </div>
            </div>
        </div>
    </div>
    """)

    initial_game_data = load_game_data()

    initial_dashboard_game_html, initial_game_html = render_game_outputs(
        initial_game_data["xp"],
        initial_game_data["missions"]
    )
    with gr.Tabs(selected="dashboard"):


        # =====================================================
        # TAB 1: Game
        # =====================================================
        with gr.Tab("🏁 Health Garden Race", id="game"):
            gr.HTML(render_game_explanation_html())
            game_tab_output = gr.HTML(initial_game_html)

        # =====================================================
        # TAB 2: Dashboard
        # =====================================================
        with gr.Tab("🏠 Dashboard", id="dashboard"):
            gr.HTML(dashboard_html)
            gr.HTML("""
            <div class="main-wrapper" style="margin-top:18px;">
                <div style="font-size:22px; font-weight:900; color:var(--green); margin-bottom:8px;">
                    🏆 Game Progress
                </div>
            </div>
            """)
            dashboard_game_output = gr.HTML(initial_dashboard_game_html)

            dashboard_sensor_summary = gr.HTML("""
            <div class="main-wrapper">
                <div class="page-card" style="margin:24px 0; text-align:center; color:var(--muted);">
                    Loading sensor data...
                </div>
            </div>
            """)
            dashboard_refresh_btn = gr.Button("🔄 Refresh Dashboard Sensors")
            build_manager_setup_section([dashboard_game_output, game_tab_output])

        # =====================================================
        # TAB 3: AI Chatbot
        # =====================================================
        with gr.Tab("💬 AI Chatbot", id="chatBot"):
            with gr.Column(elem_classes=["main-wrapper"]):
                gr.HTML("""
                <div class="page-card">
                    <div class="page-title">Basil AI Chatbot</div>
                    <div class="page-subtitle">
                        Ask plant-care questions in a chat format. The bot uses Gemini AI first. If Gemini is unavailable, it uses prepared chatbot patterns.
                    </div>
                </div>
                """)
                gr.HTML("""
                <div class="info-mini-card" style="margin-bottom:12px;">
                    <b>How it works:</b> Gemini AI answers first. If Gemini is unavailable or fails, the chatbot returns prepared pattern-based answers. This chatbot is separate from RAG; Academic Search still uses the project papers.
                </div>
                """)
                chatbot_ui = gr.Chatbot(
                    label="Basil AI Bot",
                    height=430,
                    elem_id="rag-chatbot-ui",
                    elem_classes=["chatbot-readable"],
                    type="messages",
                )
                with gr.Row():
                    chat_input = gr.Textbox(
                        placeholder="Ask about basil care, watering, sensors, mildew, humidity, temperature, or leaf symptoms...",
                        show_label=False,
                        scale=4,
                    )
                    chat_submit_btn = gr.Button("Send 🚀", variant="primary", scale=1)

        # =====================================================
        # TAB 4: Plant Scanner
        # =====================================================
        with gr.Tab("📷 Plant Scanner",id="plantScanner"):
            with gr.Column(elem_classes=["main-wrapper"]):
                gr.HTML("""
                <div class="page-card">
                    <div class="page-title">Plant Scanner</div>
                    <div class="page-subtitle">Upload basil images. Gemini is used first; if it is unavailable, the trained local Keras classifier runs automatically.</div>
                </div>
                """)

                plant_images = gr.File(
                    label="Upload basil plant images",
                    file_count="multiple",
                    file_types=["image"],
                    elem_classes=["custom-file"],
                    height=120,
                )
                upload_status = gr.HTML(update_upload_preview([])[0])
                image_preview = gr.HTML("")
                analyze_btn = gr.Button(
                    "🚀 Run AI Analysis",
                    variant="primary",
                    elem_classes=["custom-run-btn"],
                )
                ai_analysis_output = gr.HTML("")

        # =====================================================
        # TAB 5: Academic Search
        # =====================================================
        with gr.Tab("📚 Academic Search",id="Search"):
            gr.HTML("""
            <div class="main-wrapper" style="margin-bottom:30px; margin-top:10px;">
                <div style="display:flex; align-items:center; gap:16px;">
                    <div style="font-size:28px;">🔍</div>
                    <div>
                        <div style="font-size:28px; font-weight:800; color:var(--green);">Academic Search Engine</div>
                        <div style="color:var(--muted);">Boolean retrieval over basil research papers</div>
                    </div>
                </div>
            </div>
            """)
            with gr.Column(elem_classes=["main-wrapper", "research-content"]):
                with gr.Column(elem_classes=["search-panel"]):
                    search_text = gr.Textbox(
                        placeholder="Examples: mildew OR peronospora | essential oil AND antimicrobial",
                        show_label=False,
                        lines=4,
                        elem_id="academic-search-input",
                    )

                    search_btn = gr.Button(
                        "Search",
                        variant="primary",
                        elem_id="academic-search-button",
                    )
                search_output = gr.HTML("""
                <div class="page-card" style="text-align:center; padding:60px 20px;">
                    <div style="font-size:64px;">📚</div>
                    <h3 style="color:var(--text);">Ready to explore</h3>
                    <p style="color:var(--muted);">Search the indexed academic database.</p>
                </div>
                """)

        # =====================================================
        # TAB 6: Live Telemetry
        # =====================================================
        with gr.Tab("📡 Live Telemetry",id="Telemetry"):
            sensor_cards = gr.HTML(make_sensor_cards_html(fetch_live=False))

            with gr.Column(elem_classes=["main-wrapper"]):
                with gr.Row():
                    temperature_btn = gr.Button("🌡️ Temperature")
                    humidity_btn = gr.Button("💧 Humidity")
                    soil_btn = gr.Button("🌱 Soil")
                    refresh_cards_btn = gr.Button("🔄 Refresh Cards", variant="primary")

                with gr.Row():
                    sensor_limit = gr.Slider(1, 100, value=10, step=1, label="Samples to display")
                    date_filter = gr.Textbox(label="Filter by Date (YYYY-MM-DD)", placeholder="e.g. 2026-06-09 (Leave empty for live data)")

                active_sensor = gr.State("temperature")

                live_output = gr.HTML("""
                <div class="page-card" style="text-align:center;">
                    <h3 style="color:var(--text);">Choose a sensor</h3>
                    <p style="color:var(--muted);">The graph and table will appear here.</p>
                </div>
                """)
                live_plot = gr.Plot(label="Sensor Graph")
                live_table = gr.Dataframe(label="Sensor Data Table (Newest first)")

    # Chatbot wiring - standalone, no game
    chat_submit_btn.click(
        fn=standalone_ai_chatbot_stream,
        inputs=[chat_input, chatbot_ui],
        outputs=[chat_input, chatbot_ui],
    )

    chat_input.submit(
        fn=standalone_ai_chatbot_stream,
        inputs=[chat_input, chatbot_ui],
        outputs=[chat_input, chatbot_ui],
    )

    # Scanner wiring
    plant_images.upload(
    update_upload_preview,
    inputs=[plant_images],
    outputs=[upload_status, image_preview],
    )

    plant_images.change(
        update_upload_preview,
        inputs=[plant_images],
        outputs=[upload_status, image_preview],
    )
    analyze_btn.click(
        gamified_image_analysis,
        inputs=[plant_images],
        outputs=[ai_analysis_output, dashboard_game_output, game_tab_output],
    )

    # Search wiring
    search_btn.click(gamified_search, inputs=[search_text], outputs=[search_output, dashboard_game_output, game_tab_output])
    search_text.submit(gamified_search, inputs=[search_text], outputs=[search_output, dashboard_game_output, game_tab_output])

    # Sensor graph update wiring
    temperature_btn.click(
        lambda limit, date: gamified_show_sensor_graph("temperature", limit, date),
        inputs=[sensor_limit, date_filter],
        outputs=[active_sensor, sensor_cards, live_output, live_plot, live_table, dashboard_game_output, game_tab_output],
    )
    humidity_btn.click(
        lambda limit, date: gamified_show_sensor_graph("humidity", limit, date),
        inputs=[sensor_limit, date_filter],
        outputs=[active_sensor, sensor_cards, live_output, live_plot, live_table, dashboard_game_output, game_tab_output],
    )
    soil_btn.click(
        lambda limit, date: gamified_show_sensor_graph("soil", limit, date),
        inputs=[sensor_limit, date_filter],
        outputs=[active_sensor, sensor_cards, live_output, live_plot, live_table, dashboard_game_output, game_tab_output],
    )
    refresh_cards_btn.click(
        gamified_live_cards_refresh,
        outputs=[sensor_cards, dashboard_game_output, game_tab_output],
    )

    sensor_limit.change(
        show_sensor_graph,
        inputs=[active_sensor, sensor_limit, date_filter],
        outputs=[active_sensor, sensor_cards, live_output, live_plot, live_table],
    )
    date_filter.change(
        show_sensor_graph,
        inputs=[active_sensor, sensor_limit, date_filter],
        outputs=[active_sensor, sensor_cards, live_output, live_plot, live_table],
    )

    dashboard_refresh_btn.click(
        gamified_sensor_refresh,
        outputs=[dashboard_sensor_summary, dashboard_game_output, game_tab_output],
    )

    app.load(
        refresh_dashboard_sensor_summary,
        outputs=[dashboard_sensor_summary],
    )
    app.load(
        refresh_game_outputs,
        outputs=[dashboard_game_output, game_tab_output],
    )

print("Frontend application with game tab, Academic RAG chatbot, and dashboard gamification constructed successfully.")

import gradio as gr
print(gr.__version__)

"""#6. QA Tests"""

# ==========================================
# QA TESTS
# ==========================================

import tempfile


def run_qa_tests():
    passed_tests = []

    def check(test_name: str, condition: bool):
        """Fail immediately when a test condition is not correct."""
        if not condition:
            raise AssertionError(f"QA test failed: {test_name}")

        passed_tests.append(test_name)

    # ==========================================
    # 1. Papers and inverted index tests
    # ==========================================

    check(
        "Five basil papers are loaded",
        len(basil_papers) == 5
    )

    check(
        "Target inverted index is not empty",
        bool(inverted_index)
    )

    check(
        "Mildew is indexed",
        "mildew" in inverted_index
    )



    # ==========================================
    # 1B. File, Firebase export, and dashboard tests
    # ==========================================

    check(
        "Papers are saved as a JSON file",
        PAPERS_FILE_PATH.exists()
    )

    loaded_papers_from_file = load_papers_from_file()

    check(
        "RAG papers are loaded dynamically from file",
        len(loaded_papers_from_file) == len(basil_papers)
        and loaded_papers_from_file[0]["title"] == basil_papers[0]["title"]
    )

    firebase_payload = build_firebase_index_payload(inverted_index)

    check(
        "Firebase payload contains term counts and DocIDs",
        "mildew" in firebase_payload
        and firebase_payload["mildew"]["count"] > 0
        and len(firebase_payload["mildew"]["DocIDs"]) > 0
    )



    # ==========================================
    # 2. Boolean search tests
    # ==========================================

    # תיקון: משתמשים במילים שקיימות ב-TARGET_TERMS של האינדקס
    and_results = boolean_search("mildew AND peronospora")

    check(
        "Boolean AND returns results",
        len(and_results) > 0
    )

    check(
        "Boolean AND returns only matching papers",
        all(
            {"mildew", "peronospora"}.issubset(
                set(result["found_words"])
            )
            for result in and_results
        )
    )

    or_results = boolean_search("mildew OR linalool")

    check(
        "Boolean OR returns results",
        len(or_results) > 0
    )

    # ==========================================
    # 3. Local RAG tests
    # ==========================================

    local_rag = EcologicalRAG(
        gemini_api_key=None,
        use_transformers=False
    )

    local_rag.load_papers(basil_papers)

    rag_result = local_rag.query(
        "What causes basil downy mildew?",
        n_results=2
    )

    check(
        "RAG returns two papers",
        rag_result["papers_found"] == 2
    )

    check(
        "RAG response is text",
        isinstance(rag_result["response"], str)
        and len(rag_result["response"].strip()) > 0
    )

    check(
        "RAG uses the non-AI extractive mode",
        rag_result["response_mode"] == "extractive"
    )

    check(
        "RAG retrieves the downy mildew paper",
        any(
            "peronospora belbahrii" in (
                paper.get("title", "")
                + " "
                + paper.get("abstract", "")
            ).casefold()
            for paper in rag_result["papers"]
        )
    )

    check(
        "RAG response includes article sources",
        "Sources used" in rag_result["response"]
        and "Source" in rag_result["response"]
    )

    # ==========================================
    # 4. Sensor status tests
    # ==========================================

    check(
        "37.5°C is correctly marked High",
        basil_agent.sensor_status("temperature", 37.5)[0] == "High"
    )

    check(
        "25°C is correctly marked Optimal",
        basil_agent.sensor_status("temperature", 25)[0] == "Optimal"
    )

    check(
        "100% soil moisture is Too Wet",
         basil_agent.sensor_status("soil", 100)[0] == "Too Wet"
    )

    check(
        "Missing sensor value is No Data",
         basil_agent.sensor_status("humidity", "--")[0] == "No Data"
    )

    # ==========================================
    # 5. Dashboard tests
    # ==========================================

    placeholder_dashboard = make_dashboard_sensor_summary_html(
        fetch_live=False
    )

    normalized_dashboard = placeholder_dashboard.casefold()

    expected_sensor_labels = [
        sensor_spec["label"].casefold()
        for sensor_spec in SENSOR_SPECS.values()
    ]

    check(
        "Dashboard renders all three sensor labels",
        all(
            label in normalized_dashboard
            for label in expected_sensor_labels
        )
    )

    check(
        "Dashboard does not append units to missing values",
        "No data°C" not in placeholder_dashboard
        and "No data%" not in placeholder_dashboard
    )
    # ==========================================
    # 6. Image upload preview tests
    # ==========================================

    empty_status, empty_gallery = update_upload_preview(None)

    check(
        "Empty upload preview is handled",
        "No Image Uploaded" in empty_status
        and "IMAGES UPLOADED" in empty_status
        and empty_gallery == ""
    )

    with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as test_image:
        test_image_path = test_image.name

    PIL.Image.new("RGB", (10, 10), color=(0, 255, 0)).save(test_image_path)

    uploaded_status, uploaded_gallery = update_upload_preview([test_image_path])

    normalized_upload_status = "".join(uploaded_status.split())

    check(
        "Successful upload displays the count and status",
        "UploadedSuccessfully" in normalized_upload_status
        and "READYTOANALYZE" in normalized_upload_status
        and ">1<" in normalized_upload_status
        and isinstance(uploaded_gallery, str)
        and "base64" in uploaded_gallery
    )


    print(f"✅ {len(passed_tests)} QA tests passed:")

    for test_name in passed_tests:
        print("  -", test_name)

run_qa_tests()

"""#7. Launch"""

app.launch(share=False, inline=True, debug=False)
